# 07: Generalisation

*How generalisable are the KG's inferences across disease domains and data regimes?*

| Sub-metric | Definition | Quantification | Scoring |
|---|---|---|---|
| **Data-sparse generalisation** | Link prediction on the least-connected diseases, where most rare conditions sit | AUROC on pairs whose disease falls in the lowest quartile of total graph degree | 2·AUROC − 1 (Gini) |
| **Cross-domain generalisation** | Consistency of link prediction across therapeutic areas | Mean AUROC over areas with at least `MIN_PAIRS` pairs; the coefficient of variation reports the spread | 2·AUROC − 1 (Gini) |
| **Prospective generalisation** | Prediction of drug-disease pairs approved after the graph was built | AUROC on post-cutoff approved pairs that are not already edges in that graph, reported with n and clustered CIs | 2·AUROC − 1 (Gini), conditional on the evaluable set |

The dimension score is the equal-weighted mean of the three, each first mapped to 2·AUROC − 1 (the Gini coefficient, equal to Somers' D) so that chance scores 0, as notebook 06 does for link prediction. Raw AUROCs are kept in `sub_scores_raw` and the tables, and are the numbers to quote.

**Reported, not scored.** Prospective coverage and full-denominator Hits@K (below), the Q4 → Q1 degradation profile, the per-area coefficient of variation, the rank distribution of prospective hits, and the network-proximity replication of Chandak et al. (2023).

**Three prospective numbers, because one is not honest on its own.** The graphs differ by a factor of sixty in how many gold pairs they can express, and the evaluable subsets are not missing at random, since an older graph holds the better-studied and therefore easier drugs. Coverage is the evaluable fraction. Hits@K (all) uses the whole gold set as its denominator, so a pair the graph cannot express counts as the failure to retrieve that it is, with nothing imputed. AUROC stays conditional on the evaluable subset, because it is defined over a ranked candidate list and a pair with no nodes has no list; inserting a zero there would assert the true positive ranked last and push the graph below AUROC's 0.5 chance floor. Only the AUROC is scored: coverage is already scored in dimension 01, and charging it again here would count one deficiency twice in Fig 1. A pairwise table compares each two graphs on the pairs both can evaluate, which is the complete-case check on whether a graph ranks better or merely drew an easier sample.

**Prospective means two things, both tested per graph.** A pair counts only if the approval post-dates the graph's content date (`config/kg_snapshots.yaml`) and no direct drug-disease edge of any relation exists in the graph being scored. Pairs failing either test are removed per graph, not globally, because a pair can be novel to one graph and asserted in another. A third, reported test flags pairs whose new disease sits within two `is_a` steps of a disease the graph already links the drug to (a line extension from the graph's point of view); `prospective_strict` in `config/params.yaml` turns that flag into a filter.

**Intervals are clustered by disease and by drug.** Several prospective pairs share a disease, a few drugs contribute many pairs, and every positive carries its own generated negatives, so rows within a disease or a drug share a candidate pool, a neighbourhood and a difficulty. Resampling rows treats them as independent and is too narrow; the row, disease-clustered and drug-clustered intervals are all recorded, and the widest is what should be reported.

**A graph is eligible only if it predates the approvals.** The post-cutoff approval set runs from 2023-01-19, so a graph whose content date falls after `prospective_max_content_date` (2022-12-31, `config/params.yaml`) already holds most of the pairs it would be tested on, and its coverage of the set measures its age rather than any ability to predict. Such a graph is excluded before any score is computed and reports N/A with the reason. On the current panel that is MATRIX, released 2026-05-28, which only 5 of the 128 approvals postdate. Content dates and the evidence for each are in `config/kg_snapshots.yaml`.

**Sub-metrics are skipped, not imputed, when their data is too thin.** A graph with fewer than `MIN_NEW_PAIRS` prospective pairs scores N/A on that axis rather than 0, because no signal and no evidence are different claims.

**Inputs:** `config/kgs.yaml` · `config/params.yaml` · KG edge and node files · `data/gold_standards/bridges/do_diseases.csv` · `data/gold_standards/bridges/mesh_to_doid.csv` · `data/gold_standards/bridges/mondo_hierarchy.csv` · `data/gold_standards/post_cutoff_approvals/post_cutoff_approvals.csv` · `data/gold_standards/post_cutoff_approvals/post_cutoff_pair_status.csv` · `config/kg_snapshots.yaml`

**Outputs:** `results/{figures,tables}/07_generalization/` · `results/checkpoints/07_generalization.pkl`

Design notes and justifications: [`README.md`](README.md)

## Set-up

All graphs, node tables, and shared utilities are loaded from the parent-level `src/` module. Link prediction uses Adamic-Adar scoring throughout, consistent with the rest of this benchmark. A fixed random seed (`seed = 42`) and negative-to-positive ratio of 5:1 are used in all sampling procedures.

In [1]:
# Pick up edits to src/ without restarting the kernel. Without this, editing a
# module after it has been imported leaves the kernel holding the old version,
# which surfaces as a confusing ImportError for a name that is plainly on disk.
try:
    get_ipython().run_line_magic('load_ext', 'autoreload')
    get_ipython().run_line_magic('autoreload', '2')
except Exception:
    pass                      # not running under IPython (headless runner, plain python)

from pathlib import Path
import gc
import importlib
import pickle
import random
import re as _re
import sys
import time
import warnings
import yaml
import zlib

warnings.filterwarnings('ignore')

# Locate the repo root from any depth, then make the package importable.
# (`pip install -e .` also works and makes this block a no-op.)
_root = next(p for p in [Path.cwd().resolve(), *Path.cwd().resolve().parents]
             if (p / 'config' / 'kgs.yaml').exists())
for _p in (str(_root / 'src'), str(Path.cwd().resolve())):
    if _p not in sys.path:
        sys.path.insert(0, _p)

from scipy import stats
import networkx as nx
import numpy as np
import pandas as pd


from biokgsuite.gold import post_cutoff_approvals as _pg
from biokgsuite.gold import regulatory_approvals as _ra
from biokgsuite.gold.ontology_classifier import ONTOLOGY_DOMAINS, OntologyClassifier
from biokgsuite.gold.regulatory_approvals import load_expanded_approvals
from biokgsuite.graph.graph_utils import build_graph, build_lookup_maps, find_node
from biokgsuite.graph.negative_sampling import generate_negatives
from biokgsuite.graph.scoring import score_adamic_adar
from biokgsuite.io import Outputs, assert_full_panel, find_config, load_config, load_kg, require
from biokgsuite.stats.evaluation import (bootstrap_auprc_ci, bootstrap_auprc_ci_clustered,
                                         bootstrap_auroc_ci, bootstrap_auroc_ci_clustered,
                                         compute_metrics)
from biokgsuite.stats.prospective import pairwise_agreement, prospective_summary
from biokgsuite.viz.dimension07 import (fig07_domain_cv, fig07_domain_heatmap,
                                        fig07_prospective_auroc, fig07_prospective_hits,
                                        fig07_proximity, fig07_rank_distribution,
                                        fig07_sparse_tiers)
from biokgsuite.viz.style import (KG_PALETTE, setup_style)

setup_style()

In [2]:
# Config and paths
config  = load_config(find_config(_root))
params  = config['analysis_params']
BASE    = config['_base_dir']
FIGS = BASE / 'results' / 'figures' / '07_generalization'; FIGS.mkdir(parents=True, exist_ok=True)
# Every other dimension writes through Outputs, which stamps the notebook name
# and reports what it produced. 07 wrote figures straight to FIGS and pickled
# its checkpoint by hand, so none of its outputs appeared in a manifest.
out = Outputs('07_generalization', BASE)

# Spine styling helper (used in all section plots)
def _style_left_spine(ax):
    ax.spines['left'].set_visible(True)
    ax.spines['left'].set_color('#333333')
    ax.spines['left'].set_linewidth(0.5)

# A run that scores fewer graphs than config/kgs.yaml declares is a
# different experiment, not a smaller one, so it has to be asked for.
ALLOW_PARTIAL_PANEL = False

# The same argument for the prospective checks. post_cutoff_pair_status.csv is
# built per graph; a graph absent from it keeps every pair that the content-date
# and leakage checks would have removed, which inflates its post-cutoff set
# rather than shrinking it. Rebuild the table for all six graphs (see this
# folder's README) instead of setting this to True.
ALLOW_MISSING_PAIR_STATUS = False

### Loading

In [3]:
# Free any KG/graph state left over from a previous notebook (prevents
# the ~150 GB OOM that hit when running notebooks back-to-back in one kernel).
for _v in ('kgs', 'graphs', 'maps', 'preps'):
    try:
        del globals()[_v]
    except KeyError:
        pass
gc.collect()

# Load KGs
KG_NAMES  = list(config['knowledge_graphs'].keys())
KG_COLORS = {n: KG_PALETTE.get(n, '#888888') for n in KG_NAMES}

kgs = {}
for name in KG_NAMES:
    try:
        kg, nodes = load_kg(name, config)
        kgs[name] = {'kg': kg, 'nodes': nodes}
        print(f'{name:12s}: {len(kg):>12,} edges  {len(nodes):>8,} nodes  '
              f'{kg["relation"].nunique():>4} relation types')
    except FileNotFoundError:
        print(f'{name:12s}: [NOT FOUND]')
    except Exception as e:
        print(f'{name:12s}: [ERROR] {e}')

LOADED = [n for n in KG_NAMES if n in kgs and config['knowledge_graphs'].get(n, {}).get('relations')]
print(f'\nLoaded: {LOADED}')
# Refuse to score a panel that is missing graphs.
#
# The broad `except` above keeps one unreadable graph from taking down the run,
# but the cost is that a run finding two of six completes, prints a dimension
# score and writes a checkpoint indistinguishable from a full one. Scores that
# intersect or normalise across the panel then differ from what a full run would
# give, silently. Set ALLOW_PARTIAL_PANEL in the config cell to score a subset
# deliberately; the reduced panel is recorded in the checkpoint either way.
PANEL = assert_full_panel(LOADED, config,
                          allow_partial=ALLOW_PARTIAL_PANEL,
                          notebook='07_generalization')


primekg     :    8,100,498 edges   129,375 nodes    30 relation types


hetionet    :    2,250,197 edges    47,031 nodes    24 relation types


drkg        :    5,874,261 edges    97,238 nodes   107 relation types


openbilink  :    4,986,521 edges   185,929 nodes    28 relation types


biokg       :    2,067,998 edges   105,524 nodes    17 relation types
  matrix: loading from cache _cache_matrix_7365ba96_kg.parquet / _cache_matrix_7365ba96_nodes.parquet


  matrix: cached load — 7,354,612 nodes, 81,149,812 edges


matrix      :   81,149,812 edges  7,354,612 nodes    91 relation types

Loaded: ['primekg', 'hetionet', 'drkg', 'openbilink', 'biokg', 'matrix']
07_generalization: panel complete, 6/6 graphs loaded


In [4]:
# Build graphs and lookup maps
graphs, maps = {}, {}
for name, data in kgs.items():
    G            = build_graph(data['kg'], data['nodes'])
    m            = build_lookup_maps(data['nodes'])
    m['node_id_map'] = dict(zip(data['nodes']['idx'], data['nodes']['id']))
    # id_to_idx bare-id lookup (WARNING PrimeKG IDs can collide across types).
    # Use uid_to_idx (type/id keyed) or filter by node_type_map for safe lookups.
    m['id_to_idx']   = dict(zip(data['nodes']['id'],  data['nodes']['idx']))
    if 'uid' in data['nodes'].columns:
        m['uid_to_idx'] = dict(zip(data['nodes']['uid'], data['nodes']['idx']))
    graphs[name] = G
    maps[name]   = m
    print(f'{name:12s}: {G.number_of_nodes():,} nodes  {G.number_of_edges():,} edges')

primekg     : 129,375 nodes  4,049,642 edges


hetionet    : 47,031 nodes  2,107,709 edges


drkg        : 97,238 nodes  4,400,766 edges


openbilink  : 185,929 nodes  3,200,327 edges


biokg       : 105,524 nodes  2,043,846 edges


matrix      : 7,354,612 nodes  54,038,933 edges


### KG configuration

In [5]:
# KG config from config file
KG_CONFIG = {name: config['knowledge_graphs'][name] for name in LOADED}

def kg_drug_type(name):       return KG_CONFIG[name]['entity_types']['Drug']
def kg_disease_type(name):    return KG_CONFIG[name]['entity_types']['Disease']
def kg_indication_rels(name):
    # Handles singular `relation` and plural `relations`. The singular-only form
    # raised KeyError on any KG whose config uses the plural key, which 06's
    # accessor already handled.
    cfg = KG_CONFIG[name]['relations']['drug_disease']
    return list(cfg['relations']) if 'relations' in cfg else [cfg['relation']]

In [6]:
# Entity sets, positive pairs, and disease degree

SEED      = require(params, 'random_seed')
# require(), not .get(): the fallback here was neg_ratio=1 while params.yaml says
# 5, so a missing or renamed key would have quietly quintupled the positive rate
# relative to every other dimension. Same fix 06 already carries.
NEG_RATIO = require(params, 'neg_ratio')


def _kg_rng(name: str, purpose: str) -> np.random.RandomState:
    """A generator keyed on the KG and on what the draw is for.

    A single generator consumed across the KG loop makes every draw depend on
    how many draws the graphs before it made, so adding a KG to config/kgs.yaml
    moves the tier samples and negatives of every KG after it, for a reason that
    has nothing to do with the graphs. Keyed this way, a graph's numbers are the
    same whether it runs alone or in a panel of six.
    """
    return np.random.RandomState(
        SEED + (zlib.crc32(f'{name}|{purpose}'.encode()) % 100_000))

kg_entity_sets, kg_pos_pairs, kg_disease_degree = {}, {}, {}

for name in LOADED:
    kg, nodes = kgs[name]['kg'], kgs[name]['nodes']
    dt, dis_t = kg_drug_type(name), kg_disease_type(name)
    ntm       = maps[name]['node_type_map']
    G         = graphs[name]

# load_config normalises entity_types to a TUPLE of labels, so every `== dt`
# below compared a Series of strings against a tuple and returned all-False.
# Nothing downstream raised; the sets were simply empty and the notebook
# produced no results at all. BioKG's Disease is ('Disease','GeneticDisorder'),
# so indexing [0] would silently drop GeneticDisorder; membership is required.
    drug_idx    = set(nodes[nodes['type'].isin(dt)   ]['idx'])
    disease_idx = set(nodes[nodes['type'].isin(dis_t)]['idx'])

    ind_edges = kg[kg['relation'].isin(kg_indication_rels(name))]
    # Vectorized - avoids iterrows() over indication edges
    if not ind_edges.empty:
        _ntm_s = pd.Series(ntm)
        _xi = ind_edges['x_index'].to_numpy()
        _yi = ind_edges['y_index'].to_numpy()
        _xt = _ntm_s.reindex(_xi).to_numpy()
        _yt = _ntm_s.reindex(_yi).to_numpy()
        _fwd  = np.isin(_xt, dt)    & np.isin(_yt, dis_t)
        _rev  = np.isin(_xt, dis_t) & np.isin(_yt, dt)
        _keep = _fwd | _rev
        _drug = np.where(_fwd, _xi, _yi)
        _dis  = np.where(_fwd, _yi, _xi)
        pos_pairs = list({(d, dz) for d, dz in zip(_drug[_keep].tolist(), _dis[_keep].tolist())})
    else:
        pos_pairs = []

    # Total graph degree for each disease node
    kg_disease_degree[name] = {
        d: G.degree(d)
        for d in disease_idx
        if G.has_node(d)
    }

    kg_entity_sets[name] = {'drug': drug_idx, 'disease': disease_idx}
    kg_pos_pairs[name]   = pos_pairs

    print(f'{name:12s}: drugs={len(drug_idx):,}  diseases={len(disease_idx):,}  '
          f'indication pairs={len(pos_pairs):,}')


primekg     : drugs=7,957  diseases=17,080  indication pairs=9,388
hetionet    : drugs=1,552  diseases=137  indication pairs=755
drkg        : drugs=24,313  diseases=5,103  indication pairs=4,968


openbilink  : drugs=78,592  diseases=9,510  indication pairs=7,164
biokg       : drugs=8,805  diseases=9,490  indication pairs=66,867


matrix      : drugs=1,752,031  diseases=125,110  indication pairs=581,112


## 1. Data-Sparse generalisation

Rare and orphan diseases are systematically under-represented in public biomedical data. Whether a KG can still support accurate inference on them is a critical test of generalisation. This section bins diseases by total graph degree (quartile Q1 = least connected, Q4 = most connected) and compares link prediction performance across tiers.

**Protocol:** For each disease quartile, AUROC and AUPRC (type-constrained negatives, Adamic-Adar heuristic) are computed per tier. Total degree measures the structural context available to topology-based heuristics, making it the natural stratification axis. Degradation from Q4 to Q1 quantifies structural prevalence bias.

In [7]:
# Disease connectivity tiers by total graph degree quartile
TIERS = ['Q1: Sparse', 'Q2: Under-studied', 'Q3: Moderate', 'Q4: Well-studied']

kg_degree_df, kg_disease_to_tier = {}, {}

for name in LOADED:
    degs = kg_disease_degree[name]
    degree_df = pd.DataFrame(
        [(d, degs[d]) for d in kg_entity_sets[name]['disease'] if d in degs],
        columns=['disease_idx', 'degree']
    )

    # Quartile binning on total degree.
    # pd.qcut with duplicates='drop' handles ties at quartile boundaries.
    try:
        cuts = pd.qcut(degree_df['degree'], q=4, duplicates='drop')
        n_cats = len(cuts.cat.categories)
        degree_df['tier'] = pd.qcut(
            degree_df['degree'], q=4, duplicates='drop',
            labels=TIERS[:n_cats]
        )
    except Exception:
        # Fallback manual quartile bins
        qs = degree_df['degree'].quantile([0.25, 0.5, 0.75])
        degree_df['tier'] = pd.cut(
            degree_df['degree'],
            bins=[-np.inf, qs[0.25], qs[0.50], qs[0.75], np.inf],
            labels=TIERS, include_lowest=True
        )

    kg_degree_df[name]       = degree_df
    kg_disease_to_tier[name] = dict(zip(degree_df['disease_idx'], degree_df['tier']))

    print(f'\n{name} - disease total-degree tiers:')
    print(degree_df.groupby('tier', observed=False)['degree']
          .agg(['count', 'min', 'max', 'mean']).round(1).to_string())



primekg - disease total-degree tiers:
                   count  min   max  mean
tier                                     
Q1: Sparse          4999    1     2   1.4
Q2: Under-studied   4044    3     6   4.1
Q3: Moderate        3886    7    21  12.5
Q4: Well-studied    4151   22  1524  64.7

hetionet - disease total-degree tiers:
                   count  min   max   mean
tier                                      
Q1: Sparse            35    0    65   39.8
Q2: Under-studied     34   67   151   99.6
Q3: Moderate          34  153   442  251.7
Q4: Well-studied      34  470  1123  692.7

drkg - disease total-degree tiers:
                   count  min   max   mean
tier                                      
Q1: Sparse          1788    1     2    1.3
Q2: Under-studied    912    3     5    3.8
Q3: Moderate        1137    6    17   10.1
Q4: Well-studied    1266   18  8116  152.5

openbilink - disease total-degree tiers:
                   count  min   max  mean
tier                             

In [8]:
# Link prediction performance by disease connectivity tier
N_BOOTSTRAP = require(params, 'n_bootstrap')
tier_results_kg = {}

for name in LOADED:
    G         = graphs[name]
    pos_pairs = kg_pos_pairs[name]
    pos_set   = set(pos_pairs)
    ents      = kg_entity_sets[name]
    nm        = maps[name]['node_name_map']
    tier_results_kg[name] = {}

    print(f'\n{name}')
    print(f'  {"Tier":<24s}  {"N pairs":>8s}  {"AUROC":>7s}  {"AUPRC":>7s}')
    print('  ' + '-' * 56)

    for tier in TIERS:
        tier_diseases = set(
            kg_degree_df[name].loc[kg_degree_df[name]['tier'] == tier, 'disease_idx'])
        tier_pos  = [(d, dis) for d, dis in pos_pairs if dis in tier_diseases]
        sample_n  = min(500, len(tier_pos))

        # Skip tier if no positive pairs exist (avoids zero-length array in compute_metrics)
        if sample_n == 0:
            tier_results_kg[name][tier] = {'n_pos': 0, 'auroc': float('nan'), 'auprc': float('nan')}
            print(f'  {tier:<24s}  {0:>8,}  {"N/A":>7s}  {"N/A":>7s}')
            continue

        _srng     = _kg_rng(name, f'tier-sample|{tier}')
        sampled   = [tier_pos[i] for i in _srng.choice(len(tier_pos), sample_n, replace=False)]
        neg       = generate_negatives(
            sampled, sample_n * NEG_RATIO, 'type-constrained',
            ents['drug'], tier_diseases, {}, nm, pos_set,
            _kg_rng(name, f'tier-neg|{tier}'))

        if not neg:
            tier_results_kg[name][tier] = {
                'n_pos': len(tier_pos), 'n_sampled': 0, 'n_neg': 0,
                'auroc': float('nan'), 'auprc': float('nan')}
            print(f'  {tier:<24s}  {len(tier_pos):>8,}  {"N/A":>7s}  {"N/A":>7s}'
                  f'  (no negatives available in this tier)')
            continue

        _scores = np.asarray(score_adamic_adar(G, sampled + neg), dtype=float)
        _labels = np.array([1] * len(sampled) + [0] * len(neg))
        m       = compute_metrics(_scores, _labels)
        # Bootstrap CIs. bootstrap_auroc_ci/bootstrap_auprc_ci were imported at
        # the top of this cell and never called, so the tier figure drew bars
        # with no uncertainty while Q1 rests on far fewer pairs than Q4 and is
        # the tier the whole dimension is about.
        #
        # The keyword is rng_, not rng. Passing rng= raises TypeError, and a
        # try/except around it would fall back to the unseeded default and
        # quietly undo the panel independence above.
        _, m['auroc_ci_lo'], m['auroc_ci_hi'] = bootstrap_auroc_ci(
            _scores, _labels, n_boot=N_BOOTSTRAP,
            rng_=_kg_rng(name, f'tier-boot|{tier}'))
        _, m['auprc_ci_lo'], m['auprc_ci_hi'] = bootstrap_auprc_ci(
            _scores, _labels, n_boot=N_BOOTSTRAP,
            rng_=_kg_rng(name, f'tier-bootprc|{tier}'))
        # n_pos is how many positives EXIST in the tier; the metric is computed
        # on at most 500 of them. Reporting only the first overstates the
        # evaluation behind the number by up to an order of magnitude.
        m['n_pos']     = len(tier_pos)
        m['n_sampled'] = len(sampled)
        m['n_neg']     = len(neg)
        tier_results_kg[name][tier] = m
        print(f'  {tier:<24s}  {len(tier_pos):>8,}  {m["auroc"]:7.4f}  {m["auprc"]:7.4f}'
              f'   (scored on {len(sampled):,} pos + {len(neg):,} neg)')



primekg
  Tier                       N pairs    AUROC    AUPRC
  --------------------------------------------------------


  Q1: Sparse                      21   0.6905   0.4841   (scored on 21 pos + 105 neg)


  Q2: Under-studied              230   0.8341   0.7152   (scored on 230 pos + 1,150 neg)


  Q3: Moderate                 1,366   0.8916   0.7890   (scored on 500 pos + 2,500 neg)


  Q4: Well-studied             7,771   0.9493   0.8058   (scored on 500 pos + 2,500 neg)

hetionet
  Tier                       N pairs    AUROC    AUPRC
  --------------------------------------------------------


  Q1: Sparse                      22   0.7736   0.6167   (scored on 22 pos + 110 neg)


... (13 outputs left out)

  Q4: Well-studied            62,993   0.8913   0.7183   (scored on 500 pos + 2,500 neg)

matrix
  Tier                       N pairs    AUROC    AUPRC
  --------------------------------------------------------


  Q1: Sparse                      64   0.8750   0.7917   (scored on 64 pos + 320 neg)


  Q2: Under-studied            2,645   0.9790   0.9650   (scored on 500 pos + 2,500 neg)


  Q3: Moderate               578,403   0.9983   0.9879   (scored on 500 pos + 2,500 neg)
  Q4: Well-studied                 0      N/A      N/A


### Generalisation gap

In [9]:
# Generalization gap relative AUROC drop from Q4 (well-studied) to Q1 (sparse)
def _safe(val, decimals):
    return round(val, decimals) if not np.isnan(val) else None

gen_gaps, rows = {}, []
for name in LOADED:
    r   = tier_results_kg[name]
    q4  = r.get('Q4: Well-studied', {}).get('auroc', np.nan)
    q1  = r.get('Q1: Sparse',       {}).get('auroc', np.nan)
    gap = (q4 - q1) / q4 * 100 if not (np.isnan(q4) or np.isnan(q1) or q4 == 0) else np.nan
    gen_gaps[name] = gap
    rows.append({
        'KG':                        name.upper(),
        'Q4 AUROC (well-studied)':   _safe(q4,  4),
        'Q1 AUROC (sparse)':         _safe(q1,  4),
        'Relative gap (%)':          _safe(gap, 1),
    })

gap_df = pd.DataFrame(rows).set_index('KG')
gap_df.style \
    .set_caption('Sparse-entity generalization gap') \
    .format(na_rep='N/A') \
    .set_table_styles([{'selector': 'caption',
                        'props': [('font-weight', 'bold'), ('font-size', '1.0em')]}])

,Q4 AUROC (well-studied),Q1 AUROC (sparse),Relative gap (%)
KG,,,
PRIMEKG,0.949300,0.690500,27.300000
HETIONET,0.867600,0.773600,10.800000
DRKG,0.948200,0.557900,41.200000
OPENBILINK,N/A,0.642000,N/A
BIOKG,0.891300,0.602700,32.400000
MATRIX,N/A,0.875000,N/A


In [10]:
# Sparse-entity generalisation: AUROC and AUPRC by connectivity tier.
out.figure(fig07_sparse_tiers(tier_results_kg, TIERS, LOADED, KG_COLORS),
           '07a_sparse_tiers');


  saved 07a_sparse_tiers.pdf and .png (600 dpi)


## 2. Cross-Domain generalisation

A KG optimised for one disease domain may fail to generalise to others. This section partitions drug-disease pairs by therapeutic area (curated from DrugBank) and computes AUROC per area. Systematic variation in performance (high CV% or significant Kruskal-Wallis *H*) indicates domain-specific bias.

**Protocol:** Performance is stratified by disease domain; CV > 10% signals meaningful inequity. Domains with <5 test pairs are excluded.

In [11]:
# Ontology bridge tables for disease name resolution
_do_df        = pd.read_csv(BASE / 'data/gold_standards/bridges/do_diseases.csv')
_doid_to_name = dict(zip(_do_df['doid'].astype(str), _do_df['mondo_name'].astype(str)))

_mesh_df = pd.read_csv(BASE / 'data/gold_standards/bridges/mesh_to_doid.csv', on_bad_lines='skip')
_mesh_df['mesh_clean'] = _mesh_df['mesh_id'].astype(str).str.split(' ').str[0]
_mesh_to_doid = (_mesh_df.drop_duplicates(subset='mesh_clean')
                 .set_index('mesh_clean')['doid'].astype(str).to_dict())

# Curated MESH supplement - high-frequency DRKG nodes absent from cross-mapping file
MESH_NAME_SUPPLEMENT = {
    'MESH:D006973': 'hypertension cardiovascular',
    'MESH:D006333': 'heart failure cardiac',
    'MESH:D001171': 'arthritis joint disease',
    'MESH:D017192': 'hepatitis c infection viral',
    'MESH:D006255': 'rhinitis allergic seasonal hay fever',
    'MESH:D010146': 'pain neuropathic chronic',
    'MESH:D003865': 'major depressive disorder',
    'MESH:D006470': 'hemorrhage cardiovascular',
    'MESH:D010003': 'osteoarthritis joint arthritis',
    'MESH:D006356': 'opioid substance dependence',
    'MESH:D020521': 'stroke cerebrovascular',
    'MESH:D003928': 'diabetic nephropathies diabetes',
    'MESH:D012221': 'rhinitis allergic perennial',
    'MESH:D019698': 'hepatitis c chronic infection viral',
    'MESH:D003866': 'depressive disorder',
    'MESH:D000341': 'agitation psychomotor anxiety',
    'MESH:D011537': 'psoriasis skin autoimmune',
    'MESH:D007319': 'leukemia cancer',
    'MESH:D059413': 'heart failure cardiac',
    'MESH:D000077': 'amyotrophic lateral sclerosis neurodegenerat',
    'MESH:D065631': 'thrombosis cardiovascular',
    'MESH:D009103': 'multiple sclerosis neurology',
    'MESH:D009765': 'obesity metabolic',
    'MESH:D006509': 'hepatitis b infection viral',
    'MESH:D007938': 'leukemia cancer',
    'MESH:D009101': 'multiple myeloma cancer',
    'MESH:D008223': 'lymphoma cancer',
    'MESH:D000755': 'anemia hematology',
    'MESH:D001289': 'attention deficit hyperactivity adhd',
    'MESH:D003920': 'diabetes mellitus metabolic diabet',
    'MESH:D006693': 'hodgkin lymphoma cancer',
    'MESH:D003324': 'coronary artery disease cardiac',
    'MESH:D003072': 'cognitive impairment dementia',
    'MESH:D016212': 'follicular lymphoma cancer neoplasm',
    'MESH:D007715': 'kidney failure renal',
    'MESH:D008103': 'liver cirrhosis hepatic',
    'MESH:D018580': 'anxiety disorder psychiatric',
    'MESH:D001714': 'bipolar disorder psychiatric',
    'MESH:D013167': 'ankylosing spondylitis arthritis autoimmune',
    'MESH:D015535': 'psoriatic arthritis autoimmune',
    'MESH:D015658': 'hiv infection',
    'MESH:D001943': 'breast carcinoma cancer',
    'MESH:D014552': 'urinary tract infection',
    'MESH:D000152': 'acne skin',
    'MESH:D001007': 'anxiety disorders psychiatric',
    'MESH:D015179': 'colorectal cancer neoplasm',
    'MESH:D008175': 'lung cancer neoplasm',
    'MESH:D011467': 'prostate cancer neoplasm',
    'MESH:D009422': 'nervous system disease neurology',
    'MESH:D007239': 'infections bacterial',
    'MESH:D005355': 'fibrosis pulmonary respiratory',
    'MESH:D008288': 'malaria infection',
    'MESH:D012559': 'schizophrenia psychiatric',
    'MESH:D004194': 'disease broad',
    'MESH:D016889': 'endometrial cancer carcinoma',
    'MESH:D019337': 'hematologic neoplasm leukemia cancer',
}

# Reverse maps - used in prospective resolution (Section 3)
_name_to_doid = (_do_df.drop_duplicates(subset='mondo_name')
                 .assign(key=_do_df['mondo_name'].str.lower().str.strip())
                 .set_index('key')['doid'].astype(str).to_dict())

_doid_to_mesh = (_mesh_df.drop_duplicates(subset='doid')
                 .set_index('doid')['mesh_clean'].astype(str).to_dict())

In [12]:
# Ontology bridge tables for disease name resolution
_do_df   = pd.read_csv(BASE / 'data/gold_standards/bridges/do_diseases.csv')
_mesh_df = pd.read_csv(BASE / 'data/gold_standards/bridges/mesh_to_doid.csv', on_bad_lines='skip')
_mesh_df['mesh_clean'] = _mesh_df['mesh_id'].astype(str).str.split(' ').str[0]

# Reverse maps — used in prospective disease resolution (Section 3)
_name_to_doid = (_do_df.drop_duplicates(subset='mondo_name')
                 .assign(key=_do_df['mondo_name'].str.lower().str.strip())
                 .set_index('key')['doid'].astype(str).to_dict())

_doid_to_mesh = (_mesh_df.drop_duplicates(subset='doid')
                 .set_index('doid')['mesh_clean'].astype(str).to_dict())

In [13]:
# Disease -> therapeutic domain, per KG, from OntologyClassifier: MONDO
# hierarchy first, MeSH C-category term matching as the fallback, unclassified
# otherwise. The classification method is counted and reported rather than
# assumed, since a domain assigned by term matching is weaker evidence than one
# from the ontology.

ALL_DOMAINS = list(ONTOLOGY_DOMAINS)
_clf = OntologyClassifier(BASE)


_MESH_CODE = _re.compile(r'^[CD]\d{6,}$')


def _to_curie(node_id: str) -> str:
    """Normalise a KG node id to the CURIE form OntologyClassifier expects.

    The classifier branches on a 'DOID:' or 'MESH:' prefix to reach the MONDO
    hierarchy. The graphs do not store ids in that form:
      Hetionet  'Disease::DOID:14227'  -> 'DOID:14227'
      BioKG     'C000600608'           -> 'MESH:C000600608'
    Passed raw, every id missed both branches and fell through to MeSH term
    matching on the disease NAME. That is why the first run classified via
    'mesh_terms' for 100% of nodes and never once via 'mondo', and why BioKG
    reached only 14.4%: string matching on names, not ontology traversal.
    """
    nid = str(node_id).strip()
    if '::' in nid:                      # 'Disease::DOID:14227'
        nid = nid.split('::', 1)[1]
    if nid.startswith(('DOID:', 'MESH:', 'MONDO:')):
        return nid
    if _MESH_CODE.match(nid):            # bare MeSH descriptor or supplemental
        return 'MESH:' + nid
    if nid.isdigit():                    # bare DOID number
        return 'DOID:' + nid
    return nid


kg_disease_to_area, kg_area_method = {}, {}
for name in LOADED:
    _nid = maps[name]['node_id_map']
    _nnm = maps[name]['node_name_map']
    d2a, how_counts = {}, {}
    for d in kg_entity_sets[name]['disease']:
        domain, how = _clf.classify(_to_curie(_nid.get(d, '')),
                                    str(_nnm.get(d, '')))
        how_counts[how] = how_counts.get(how, 0) + 1
        if domain:
            d2a[d] = domain
    kg_disease_to_area[name] = d2a
    kg_area_method[name]     = how_counts
    n_dis = len(kg_entity_sets[name]['disease'])
    print(f'{name:12s}: {len(d2a):,}/{n_dis:,} diseases classified '
          f'({100 * len(d2a) / max(n_dis, 1):.1f}%)  '
          + '  '.join(f'{k}={v:,}' for k, v in sorted(how_counts.items())))

print(f'\n{len(ALL_DOMAINS)} domains: {", ".join(ALL_DOMAINS)}')


primekg     : 7,613/17,080 diseases classified (44.6%)  mesh_terms=4,375  mondo=3,238  unclassified=9,467
hetionet    : 124/137 diseases classified (90.5%)  mesh_terms=3  mondo=121  unclassified=13
drkg        : 2,063/5,103 diseases classified (40.4%)  mondo=2,063  unclassified=3,040
openbilink  : 7,574/9,510 diseases classified (79.6%)  mondo=7,574  unclassified=1,936


biokg       : 2,917/9,490 diseases classified (30.7%)  mesh_terms=716  mondo=2,201  unclassified=6,573


matrix      : 36,921/125,110 diseases classified (29.5%)  mesh_terms=32,625  mondo=4,296  unclassified=88,189

9 domains: Oncology, Infectious, Psychiatry, Neurology, Cardiovascular, Metabolic, Immunology, Respiratory, Musculoskeletal


### Evaluation

In [14]:
# Per-area evaluation config.
#
# This cell used to redefine bootstrap_auroc_ci, shadowing the package function
# imported above. The notebook therefore ran TWO different bootstraps: the tier
# and prospective sections used biokgsuite.stats.evaluation, the per-area
# section used this local copy, and nothing said so. They differ in ways that
# change the numbers:
#   - the local copy returned mean(boots) as the point estimate, a bootstrap
#     mean, where the package returns the observed statistic;
#   - it resampled unstratified, where the package preserves class balance;
#   - it drew from the shared global rng, so its intervals moved when another
#     KG was added to the panel.
# Removed. One bootstrap, used everywhere, seeded per KG.
N_BOOTSTRAP = require(params, 'n_bootstrap')
MIN_PAIRS   = require(params, 'min_pairs_per_area')


In [15]:
# Per-area link prediction performance
area_results_kg, area_cv = {}, {}

for name in LOADED:
    G, pos_pairs = graphs[name], kg_pos_pairs[name]
    d2a          = kg_disease_to_area[name]
    drug_idx     = kg_entity_sets[name]['drug']
    nm           = maps[name]['node_name_map']
    pos_set      = set(pos_pairs)
    area_results_kg[name] = {}
    kw_groups, kw_labels  = [], []

    print(f'\n{name}')
    print(f'  {"Area":<16s}  {"Diseases":>9s}  {"Pairs":>7s}  '
          f'{"AUROC":>7s}  {"95% CI":>18s}  {"AUPRC":>7s}')
    print('  ' + '-' * 78)

    for area in ALL_DOMAINS:
        area_diseases = {d for d, a in d2a.items() if a == area}
        area_pos      = [(d, dis) for d, dis in pos_pairs if dis in area_diseases]
        n_d, n_p      = len(area_diseases), len(area_pos)

        if n_p < MIN_PAIRS:
            area_results_kg[name][area] = {
                'n_diseases': n_d, 'n_pos': n_p,
                'auroc': np.nan, 'auprc': np.nan, 'ci_lo': np.nan, 'ci_hi': np.nan,
            }
            print(f'  {area:<16s}  {n_d:>9,}  {n_p:>7,}  '
                  f'{"N/A":>7s}  {"(insufficient pairs)":>18s}  {"N/A":>7s}')
            continue

        sample_n   = min(500, n_p)
        sampled    = [area_pos[i] for i in
                      _kg_rng(name, f'area-sample|{area}').choice(
                          n_p, sample_n, replace=False)]
        neg        = generate_negatives(sampled, sample_n * NEG_RATIO, 'type-constrained',
                                        drug_idx, area_diseases, {}, nm, pos_set,
                                        _kg_rng(name, f'area-neg|{area}'))
        scores_arr = np.array(score_adamic_adar(G, sampled + neg))
        labels_arr = np.array([1] * len(sampled) + [0] * len(neg))

        m             = compute_metrics(scores_arr.tolist(), labels_arr.tolist())
        _, ci_lo, ci_hi = bootstrap_auroc_ci(
            scores_arr, labels_arr, n_boot=N_BOOTSTRAP,
            rng_=_kg_rng(name, f'area-boot|{area}'))
        # n_pos counts the area's positives; the metric is computed on at most
        # 500 of them, so both are recorded rather than only the larger.
        m.update({'n_diseases': n_d, 'n_pos': n_p, 'n_sampled': len(sampled),
                  'n_neg': len(neg), 'ci_lo': ci_lo, 'ci_hi': ci_hi})
        area_results_kg[name][area] = m

        kw_groups.append(scores_arr[:len(sampled)].tolist())
        kw_labels.append(area)

        ci_str = f'[{ci_lo:.3f}, {ci_hi:.3f}]' if not np.isnan(ci_lo) else 'N/A'
        print(f'  {area:<16s}  {n_d:>9,}  {n_p:>7,}  '
              f'{m["auroc"]:7.4f}  {ci_str:>18s}  {m["auprc"]:7.4f}')

    vals = [r['auroc'] for r in area_results_kg[name].values()
            if not np.isnan(r.get('auroc', np.nan))]

    if len(vals) >= 2:
        cv  = np.std(vals) / np.mean(vals) * 100
        area_cv[name] = cv
        kw_note = ''
        if len(kw_groups) >= 2:
            kw_stat, kw_p = stats.kruskal(*kw_groups)
            sig     = '***' if kw_p < 0.001 else ('**' if kw_p < 0.01 else ('*' if kw_p < 0.05 else 'ns'))
            kw_note = f'  Kruskal-Wallis H = {kw_stat:.2f}, p = {kw_p:.4f} ({sig})'
        print(f'\n  Mean AUROC: {np.mean(vals):.4f}  SD: {np.std(vals):.4f}  '
              f'CV: {cv:.1f}%  Range: {max(vals) - min(vals):.4f}')
        if kw_note:
            print(kw_note)
    else:
        area_cv[name] = np.nan
        print('\n  [Insufficient areas with >= 25 pairs for variance summary]')


primekg
  Area               Diseases    Pairs    AUROC              95% CI    AUPRC
  ------------------------------------------------------------------------------


  Oncology              4,023    2,878   0.9545      [0.942, 0.967]   0.8888


  Infectious              722      870   0.9360      [0.921, 0.950]   0.8327


  Psychiatry              149      167   0.9860      [0.974, 0.996]   0.9607


  Neurology               824      487   0.9633      [0.951, 0.974]   0.8841


  Cardiovascular          485      449   0.9521      [0.939, 0.965]   0.8947


... (41 outputs left out)

  Metabolic             1,828   22,632   0.9964      [0.993, 0.999]   0.9906


  Immunology            2,098   21,965   0.9988      [0.997, 1.000]   0.9972


  Respiratory             467   10,240   1.0000      [1.000, 1.000]   0.9999


  Musculoskeletal       1,244    5,547   0.9988      [0.996, 1.000]   0.9971

  Mean AUROC: 0.9984  SD: 0.0014  CV: 0.1%  Range: 0.0036
  Kruskal-Wallis H = 71.77, p = 0.0000 (***)


In [16]:
# Cross-KG variance summary
print(f'{"KG":12s}  {"Mean AUROC":>12s}  {"SD":>6s}  {"CV%":>6s}  {"N areas":>7s}')
print('-' * 54)
for name in LOADED:
    vals = [r['auroc'] for r in area_results_kg[name].values()
            if not np.isnan(r.get('auroc', np.nan))]
    if vals:
        print(f'{name:12s}  {np.mean(vals):12.4f}  {np.std(vals):6.4f}  '
              f'{area_cv.get(name, np.nan):6.1f}  {len(vals):>7d}')
    else:
        print(f'{name:12s}  {"N/A":>12s}  {"N/A":>6s}  {"N/A":>6s}  {"0":>7s}')

KG              Mean AUROC      SD     CV%  N areas
------------------------------------------------------
primekg             0.9583  0.0141     1.5        9
hetionet            0.8775  0.0708     8.1        7
drkg                0.9351  0.0261     2.8        8
openbilink          0.7838  0.0895    11.4        8
biokg               0.8882  0.0355     4.0        9
matrix              0.9984  0.0014     0.1        9


### Visualisation

In [17]:
out.figure(fig07_domain_heatmap(area_results_kg, ALL_DOMAINS, LOADED),
           '07b_domain_heatmap');


  saved 07b_domain_heatmap.pdf and .png (600 dpi)


In [18]:
out.figure(fig07_domain_cv(area_cv, [n for n in LOADED if area_results_kg.get(n)],
                          KG_COLORS),
           '07c_domain_cv');


  saved 07c_domain_cv.pdf and .png (600 dpi)


### Ontology-based classification methods

Disease nodes were classified into therapeutic domains using a two-tier ontology-based approach implemented in `src/biokgsuite/gold/ontology_classifier.py`:

**Tier 1, MONDO hierarchy traversal (gold standard).** Each disease node's identifier is resolved to a MONDO ID (via DOID or MeSH cross-references in `do_diseases.csv` and `mesh_to_doid.csv`). The MONDO ontology hierarchy (`mondo_hierarchy.csv`) is then traversed upward to identify the nearest top-level disease category, mapped to one of 9 therapeutic domains (Infectious, Oncology, Cardiovascular, Neurology, Metabolic, Immunology, Respiratory, Musculoskeletal, Psychiatry). This tier provides the most reliable classification because it uses the actual ontology graph structure.

**Tier 2, MeSH C-category term matching (fallback).** For diseases not resolvable via MONDO, curated keyword lists derived from MeSH Category C subcategory scope notes are matched against the disease name. Keywords are priority-ordered following MeSH primary tree assignment conventions.

This approach replaces the prior MeSH NLM API + ad-hoc keyword method, offering three advantages: (1) reproducibility without external API dependency, (2) gold-standard ontology grounding via MONDO hierarchy, and (3) transparent classification provenance (each assignment reports whether it was resolved via `mondo` or `mesh_terms`).

**Domain size reporting.** The number of diseases and indication pairs per domain are reported above, with domains containing fewer than 25 indication pairs excluded from AUROC/AUPRC computation to avoid unreliable estimates.

## 3. Prospective generalisation

A prospective evaluation asks whether a graph's topology carries signal for
drug-disease pairs that were approved after the graph's data was assembled.
The post-cutoff approval set is `data/gold_standards/post_cutoff_approvals/post_cutoff_approvals.csv`, built
by `tools/build_post_cutoff_approvals.py` from a regulator-derived table of
new-indication decisions (FDA efficacy supplements of sub-type New Indication,
CHMP extensions of indication, Health Canada SNDS, PMDA new indications, TGA
EOI registrations; 2023-01-01 to 2026-09-17). Every row cites the regulator
document it came from; drug and disease identifiers were resolved through the
NCATS Translator services and reviewed. The construction funnel and the
per-graph checks are in `data/POST_CUTOFF_APPROVALS.md` and drawn in
`07h_post_cutoff_funnel`.

**Protocol.** For each drug that maps into a graph, every new indication is
scored with the Adamic-Adar heuristic against type-constrained negatives, with
no retraining. Metrics are AUROC, MRR and Hits@K, with intervals clustered by
disease and by drug.


### Four checks before a pair is evaluable

1. **Prospective by date.** The approval post-dates the graph's content date
   (`config/kg_snapshots.yaml`; unverified dates are printed by the builder).
2. **Covered and not leaked.** Drug and disease both resolve to nodes (degree of
   the drug node at least 5) and the graph holds no direct edge between them of
   any relation type.
3. **A new disease, not a line extension.** The new MONDO term is more than two
   `is_a` steps from every disease the graph already links the drug to. Reported
   by default; `prospective_strict: true` makes it a filter.
4. **One scorer, rank metrics, clustered intervals, common support.** The same
   heuristic scores every graph; AUROC and Hits@K are reported per graph on its
   own evaluable pairs and, when `prospective_common_support: true`, on the pairs
   evaluable in every loaded graph, so coverage and quality are not confounded.

Coverage itself is a result: the number of gold pairs a graph can even be asked
about is printed per graph and written to the checkpoint.


In [19]:
# Load FDA novel drug approvals (June 2021 - April 2026)
#
# fda_drugs_id_mapping.csv is hand-curated and is currently ABSENT from
# data/gold_standards/legacy/. It used to be read unconditionally, so its absence took
# down the whole notebook at this cell and with it sections 1 and 2, which do
# not depend on it at all.
#
# The EMA/PMDA/TGA half of the post-cutoff approval set lives inside
# regulatory_approvals.py and needs no file, so the section degrades to those
# agencies instead of failing. The loss is stated loudly here and recorded in
# the checkpoint, because a prospective result computed on 21 non-FDA pairs is
# a different and much weaker claim than one computed on the full set, and
# nothing downstream would otherwise reveal which of the two it is.
_FDA_PATH = BASE / 'data/gold_standards/legacy/fda_drugs_id_mapping.csv'
FDA_AVAILABLE = _FDA_PATH.exists()

if not FDA_AVAILABLE:
    print('=' * 72)
    print('WARNING: fda_drugs_id_mapping.csv not found at')
    print(f'  {_FDA_PATH}')
    print('The prospective evaluation will run on EMA/PMDA/TGA approvals ONLY.')
    print('Every prospective number below is therefore based on a small subset')
    print('of the intended gold standard. Restore the file before submission.')
    print('=' * 72)
    _fda_raw = pd.DataFrame(columns=['No', 'Combo_Flag', 'DrugBank_ID',
                                     'Active Ingredient', 'Approval Date',
                                     'Indication', 'Drug Name (Brand)'])
else:
    _fda_raw = pd.read_csv(_FDA_PATH, dtype=str)
print(f'Raw FDA drug list: {len(_fda_raw)} entries')

# Exclusions
# 1. Combo-flagged drugs (matched only on pre-existing component)
_combo_mask = _fda_raw['Combo_Flag'].notna() & (_fda_raw['Combo_Flag'].str.len() > 0)
# 2. #203 Kygevvi (DB02594) , ambiguous nucleoside mapping
_excl_mask  = _combo_mask | (_fda_raw['No'] == '203')
_fda = _fda_raw[~_excl_mask].copy()
print(f'After excluding {_excl_mask.sum()} combo/ambiguous entries: {len(_fda)} drugs')

# Drop rows without DrugBank ID (can't look up in KGs)
_fda = _fda[_fda['DrugBank_ID'].notna()].copy()
print(f'With DrugBank ID: {len(_fda)} drugs')

# Non-disease indications to exclude
# Imaging agents, diagnostics, cosmetics, and symptom-only entries with no
# meaningful disease node in any KG.
NON_DISEASE_INDICATIONS = {
    'MRI contrast agent',
    'Radiographic contrast agent',
    'Prostate cancer PET imaging',
    'Myocardial ischemia/infarction (PET)',
    'Optical imaging for cancer detection',
    'Ovarian cancer lesion identification',
    'Pulmonary function evaluation',
    'Moderate-to-severe glabellar lines',  # cosmetic (botox)
    'Thermal burn eschar removal',  # procedure, not disease
}
_fda = _fda[~_fda['Indication'].isin(NON_DISEASE_INDICATIONS)].copy()
print(f'After excluding non-disease indications: {len(_fda)} drugs')

# Build NEW_INDICATION_APPROVALS dict
# Format {drugbank_id (drug_name, [(disease_term, approval_date, notes), ...])}
NEW_INDICATION_APPROVALS = {}
for _, row in _fda.iterrows():
    dbid = row['DrugBank_ID']
    name = row['Active Ingredient']
    date = row['Approval Date']
    ind  = row['Indication']
    note = f"{row['Drug Name (Brand)']} — {ind}"
    if dbid not in NEW_INDICATION_APPROVALS:
        NEW_INDICATION_APPROVALS[dbid] = (name, [])
    NEW_INDICATION_APPROVALS[dbid][1].append((ind, date, note))

n_drugs = len(NEW_INDICATION_APPROVALS)
n_pairs = sum(len(v[1]) for v in NEW_INDICATION_APPROVALS.values())
print(f'\n{n_drugs} unique drugs, {n_pairs} drug–indication pairs')
# Preview
for dbid, (name, inds) in list(NEW_INDICATION_APPROVALS.items())[:3]:
    print(f'  {dbid} ({name}): {len(inds)} indication(s)')

# Expanded gold standard EMA / PMDA / TGA approvals
# Addresses reviewer concern 1.3 prospective evaluation was underpowered
# with only FDA approvals. Adding multi-agency data increases evaluable pairs.

try:
    if FDA_AVAILABLE:
        _expanded_df = load_expanded_approvals(BASE)
    else:
        # load_expanded_approvals() drops every EMA/PMDA/TGA pair that also
        # appears in the module's own 32-record FDA list, to avoid double
        # counting when the FDA CSV is merged in. With the CSV ABSENT there is
        # nothing to double count, and that filter was throwing away most of
        # the only data left: 64 non-FDA records collapse to 21, and 43 unique
        # (drug, disease) pairs collapse to 15. Below MIN_NEW_PAIRS once
        # resolution losses are taken off, which is why both graphs scored N/A.
        _expanded_df = _ra._tuples_to_df(
            _ra.EMA_APPROVALS + _ra.PMDA_APPROVALS + _ra.TGA_APPROVALS
        ).rename(columns={'drug_id': 'drugbank_id', 'disease_id': 'doid'})
        _expanded_df = _expanded_df.drop_duplicates(subset=['drugbank_id', 'doid'])
        print(f'FDA set absent: using all {len(_expanded_df)} unique non-FDA '
              f'pairs rather than only those net-new against it.')
    _n_expanded = len(_expanded_df)
    print(f'\nExpanded gold standard (EMA/PMDA/TGA): {_n_expanded} additional approval pairs')

    # Merge with FDA approvals, deduplicating by (drugbank_id, doid)
    for _, row in _expanded_df.iterrows():
        dbid = row['drugbank_id']
        dname = row['drug_name']
        disease = row['disease_name']
        date  = row.get('approval_date', '2023-01-01')
        note  = f"{row.get('source_agency', 'EMA/PMDA/TGA')} — {disease}"
        doid  = row.get('doid', '')

        if dbid not in NEW_INDICATION_APPROVALS:
            NEW_INDICATION_APPROVALS[dbid] = (dname, [])
        # Avoid duplicates (same drug + same disease term)
        existing_diseases = {t[0].lower() for t in NEW_INDICATION_APPROVALS[dbid][1]}
        if disease.lower() not in existing_diseases:
            NEW_INDICATION_APPROVALS[dbid][1].append((disease, date, note))

    _total_pairs = sum(len(v[1]) for v in NEW_INDICATION_APPROVALS.values())
    print(f'Total prospective approval pairs after merge: {_total_pairs}')
    print(f'Total unique drugs: {len(NEW_INDICATION_APPROVALS)}')

except Exception as e:
    print(f'Warning: Could not load expanded gold standard: {e}')
    print('Proceeding with FDA-only approvals.')


Raw FDA drug list: 214 entries
After excluding 11 combo/ambiguous entries: 203 drugs
With DrugBank ID: 200 drugs
After excluding non-disease indications: 191 drugs

191 unique drugs, 191 drug–indication pairs
  DB12471 (ibrexafungerp): 1 indication(s)
  DB12274 (aducanumab-avwa): 1 indication(s)
  DB08886 (asparaginase erwinia chrysanthemi (recombinant)-rywn): 1 indication(s)

Expanded gold standard (EMA/PMDA/TGA): 21 additional approval pairs
Total prospective approval pairs after merge: 206
Total unique drugs: 199


In [20]:
# Disease name resolution helpers for prospective evaluation
# (uses _name_to_doid and _doid_to_mesh built in the ontology bridge cell above)

# Comprehensive alias map FDA indication term -> DO disease name
# Maps clinical indication strings from the FDA CSV to Disease Ontology names
# that exist in _name_to_doid. Curated manually from DO cross-references.
TEMPORAL_DISEASE_ALIASES = {
    # Oncology
    'alk-positive nsclc':                       'non-small cell lung carcinoma',
    'nsclc':                                    'non-small cell lung carcinoma',
    'nsclc (egfr exon 20 insertion)':           'non-small cell lung carcinoma',
    'her2-mutated nsclc':                       'non-small cell lung carcinoma',
    'kras g12c-mutated nsclc':                  'non-small cell lung carcinoma',
    'ros1-positive nsclc':                      'non-small cell lung carcinoma',
    'c-met overexpressing nsclc':               'non-small cell lung carcinoma',
    'nsclc and pancreatic adenocarcinoma (nrg1 fusion)': 'non-small cell lung carcinoma',
    'extensive stage small cell lung cancer':   'small cell lung carcinoma',
    'bladder cancer':                           'urinary bladder cancer',
    'cutaneous squamous cell carcinoma':        'skin squamous cell carcinoma',
    'gastric/gej adenocarcinoma':               'gastric cancer',
    'her2-positive biliary tract cancer':       'biliary tract cancer',
    'er+/her2- esr1-mutated advanced breast cancer': 'breast cancer',
    'hr+/her2- metastatic breast cancer':       'breast cancer',
    'locally advanced/metastatic breast cancer': 'breast cancer',
    'breast cancer (akt pathway)':              'breast cancer',
    'kras-mutated recurrent low-grade serous ovarian cancer': 'ovarian cancer',
    'recurrent platinum-resistant ovarian cancer': 'ovarian cancer',
    'metastatic castration-resistant prostate cancer': 'castration-resistant prostate carcinoma',
    'recurrent/metastatic cervical cancer':     'cervical cancer',
    'recurrent/metastatic nasopharyngeal carcinoma': 'nasopharyngeal carcinoma',
    'refractory metastatic colorectal cancer':  'colorectal cancer',
    'unresectable hepatocellular carcinoma':    'hepatocellular carcinoma',
    'unresectable/metastatic melanoma':         'melanoma',
    'unresectable/metastatic uveal melanoma':   'uveal melanoma',
    # merkel cell carcinoma no exact DO entry; relies on KG substring match
    'diffuse midline glioma (h3 k27m)':        'brain glioma',
    'grade 2 astrocytoma / oligodendroglioma':  'oligodendroglioma',
    'pediatric low-grade glioma':               'brain glioma',
    'desmoid tumors':                           'desmoid tumor',
    'multiple solid tumor indications':         'cancer',
    'relapsed/refractory aml (idh1 mutation)':  'acute myeloid leukemia',
    'relapsed/refractory aml (npm1 mutation)':  'acute myeloid leukemia',
    'newly diagnosed aml (flt3-itd)':           'acute myeloid leukemia',
    'aml/mds (hsct preparative regimen)':       'acute myeloid leukemia',
    'relapsed/refractory dlbcl':                'diffuse large B-cell lymphoma',
    'relapsed/refractory follicular lymphoma':  'follicular lymphoma',
    'relapsed/refractory mantle cell lymphoma': 'mantle cell lymphoma',
    'relapsed/refractory multiple myeloma':     'plasma cell myeloma',  # DO name for MM
    'hematopoietic stem cell mobilization (multiple myeloma)': 'plasma cell myeloma',
    'relapsed/refractory acute leukemia':       'acute leukemia',
    'philadelphia chromosome-positive cml':     'chronic myelogenous leukemia, bcr-abl1 positive',
    'low-to-intermediate risk myelodysplastic syndromes': 'myelodysplastic syndrome',

    # Haematology (non-malignant)
    'myelofibrosis':                            'primary myelofibrosis',
    'myelofibrosis with anemia':                'primary myelofibrosis',
    'polycythemia vera':                        'polycythemia',  # DO has 'polycythemia' not 'vera'
    'immune thrombocytopenia':                  'autoimmune thrombocytopenic purpura',
    'cold agglutinin disease':                  'autoimmune hemolytic anemia',  # closest DO parent
    'hemophilia a and b':                       'hemophilia',
    'hemophilia a or b':                        'hemophilia',
    'hemolytic anemia in pyruvate kinase deficiency': 'pyruvate kinase deficiency of red cells',
    'febrile neutropenia prevention':           'neutropenia',
    'hsct-associated thrombotic microangiopathy': 'thrombotic thrombocytopenic purpura',  # closest DO

    # Immunology / Inflammation
    'moderate-to-severe atopic dermatitis':     'atopic eczema',
    'moderate-to-severe plaque psoriasis':      'psoriasis',
    'plaque psoriasis':                         'psoriasis',
    'generalized pustular psoriasis':           'psoriasis',  # DO lacks specific GPP entry
    'chronic hand eczema':                      'atopic eczema',  # closest DO
    'anca-associated vasculitis':               'vasculitis',  # DO parent
    'chronic graft-versus-host disease':        'autoimmune disease',  # no specific GVHD in DO
    'chronic spontaneous urticaria':            'urticaria',
    'relapsing multiple sclerosis':             'relapsing-remitting multiple sclerosis',
    'severe alopecia areata':                   'alopecia areata',
    'severe asthma':                            'asthma',
    'severe eosinophilic asthma':               'asthma',
    # prurigo nodularis no exact DO entry; relies on KG substring match

    # Cardiovascular / Metabolic
    'heart failure':                            'congestive heart failure',
    'obstructive hypertrophic cardiomyopathy':  'familial hypertrophic cardiomyopathy',
    'symptomatic obstructive hcm':              'familial hypertrophic cardiomyopathy',
    'transthyretin-mediated amyloid cardiomyopathy': 'familial amyloid neuropathy',  # closest TTR
    'pulmonary arterial hypertension':          'pulmonary hypertension',
    'hypertension':                             'hypertensive disorder',
    # supraventricular tachycardia no exact DO entry; relies on KG substring match
    # paroxysmal supraventricular tachycardia same
    'hypercholesterolemia / hefh':              'familial hypercholesterolemia',
    'familial hypercholesterolemia / ascvd':    'familial hypercholesterolemia',
    'nash with moderate-to-advanced liver fibrosis': 'metabolic dysfunction-associated steatohepatitis',

    # Nephrology
    'iga nephropathy':                          'IgA glomerulonephritis',
    'chronic kidney disease with type 2 diabetes': 'chronic kidney disease',
    'anemia in ckd':                            'anemia',
    'anemia in ckd (dialysis)':                 'anemia',

    # Neurology / Psychiatry
    "alzheimer's disease":                      'alzheimer disease',
    'migraine':                                 'migraine disorder',
    'episodic migraine prevention':             'migraine disorder',
    # insomnia no exact DO entry; relies on KG substring match
    'major depressive disorder':                'major depressive disorder',
    'postpartum depression':                    'postpartum depression',
    'schizophrenia':                            'schizophrenia',
    "friedreich's ataxia":                      "Friedreich ataxia",
    'als (sod1 mutation)':                      'amyotrophic lateral sclerosis',
    'amyotrophic lateral sclerosis (als)':      'amyotrophic lateral sclerosis',
    'cdkl5 deficiency disorder seizures':       'epilepsy',  # closest DO parent

    # Infectious disease
    'candidemia / invasive candidiasis':        'candidiasis',
    'hiv (multidrug-resistant)':                'hiv infectious disease',
    'post-transplant cmv infection':            'cytomegalovirus retinitis',  # closest DO CMV
    'rsv prevention in infants':                'respiratory syncytial virus infectious disease',
    'rsv prevention in neonates/infants':       'respiratory syncytial virus infectious disease',
    'mild-to-moderate covid-19':                'COVID-19',
    'h. pylori infection':                      'gastritis',  # closest DO for H. pylori
    'molluscum contagiosum':                    'molluscum contagiosum',
    'uncomplicated urinary tract infections':   'urinary tract infection',
    'complicated urinary tract infections':     'urinary tract infection',
    'uncomplicated urogenital gonorrhea':       'gonorrhea',
    'bloodstream infections / bacterial pneumonia': 'bacterial pneumonia',
    'catheter-related bloodstream infections (hemodialysis)': 'bacterial infectious disease',  # broad
    'hospital-acquired bacterial pneumonia (acinetobacter)': 'bacterial pneumonia',
    'human african trypanosomiasis':            'human african trypanosomiasis',

    # Rare / genetic diseases
    'acid sphingomyelinase deficiency':         'niemann-pick disease',
    'niemann-pick disease type c':              'niemann-pick disease',
    'activated pi3k delta syndrome':            'immunodeficiency 61',  # DO PI3K-delta == APDS
    'chaple disease (cd55 deficiency)':         'complement deficiency',
    'whim syndrome':                            'whim syndrome 1',  # DO has subtypes only
    # late-onset pompe disease no exact DO entry; relies on KG substring match
    'hereditary transthyretin-mediated amyloidosis polyneuropathy': 'familial amyloid neuropathy',
    'hereditary angioedema (acute attacks)':    'hereditary angioedema',
    'hereditary angioedema prevention':         'hereditary angioedema',
    'classic congenital adrenal hyperplasia':   'congenital adrenal hyperplasia',
    'alagille syndrome pruritus':               'Alagille syndrome',
    'pruritus (pfic)':                          'progressive familial intrahepatic cholestasis',
    'cystic fibrosis':                          'cystic fibrosis',
    'duchenne muscular dystrophy':              'Duchenne muscular dystrophy',
    'epidermolysis bullosa wounds':             'epidermolysis bullosa',
    'sepiapterin-responsive pku':               'phenylketonuria',
    'achondroplasia':                           'achondroplasia',
    'fibrodysplasia ossificans progressiva':    'fibrodysplasia ossificans progressiva',
    'neurofibromatosis type 1 (plexiform neurofibromas)': 'neurofibromatosis type 1',
    'rett syndrome':                            'Rett syndrome',
    'thymidine kinase 2 deficiency':            'mitochondrial DNA depletion syndrome',
    'barth syndrome':                           'Barth syndrome',
    'alpha-mannosidosis':                       'alpha-mannosidosis',
    'fabry disease':                            'Fabry disease',

    # Ophthalmology
    'dry eye disease':                          'dry eye syndrome',
    'demodex blepharitis':                      'blepharitis',
    'geographic atrophy (amd)':                 'age-related macular degeneration',
    'neovascular amd / diabetic macular edema': 'diabetic macular edema',
    'open-angle glaucoma / ocular hypertension': 'open-angle glaucoma',
    'presbyopia':                               'presbyopia',

    # Other
    'delay onset of stage 3 type 1 diabetes':   'type 1 diabetes mellitus',
    'growth failure (gh deficiency)':           'isolated growth hormone deficiency type ia',
    'short stature (growth hormone deficiency)': 'isolated growth hormone deficiency type ia',
    'idiopathic pulmonary fibrosis':            'pulmonary fibrosis',
    'non-cystic fibrosis bronchiectasis':       'bronchiectasis',
    # primary axillary hyperhidrosis no exact DO entry; relies on KG substring match
    # moderate-to-severe acute pain no exact DO entry; relies on KG substring match
    'motion sickness-associated vomiting':      'motion sickness',
    'familial chylomicronemia syndrome':        'familial chylomicronemia syndrome',
    'menopausal hot flashes':                   'premature menopause',  # closest DO entry
    'menopausal vasomotor symptoms':            'premature menopause',
    'hepatorenal syndrome':                     'hepatorenal syndrome',
    'pruritus in chronic kidney disease':       'chronic kidney disease',
    'chronic obstructive pulmonary disease':     'chronic obstructive pulmonary disease',
}

def _resolve_drkg_disease(term: str, drkg_dis_rev: dict):
    for t in [term.lower(), TEMPORAL_DISEASE_ALIASES.get(term.lower(), '')]:
        if not t:
            continue
        doid = _name_to_doid.get(t)
        if not doid:
            continue
        doid_node = 'DOID:' + doid.split(':')[1] if ':' in doid else None
        if doid_node and doid_node in drkg_dis_rev:
            return drkg_dis_rev[doid_node]
        mesh = _doid_to_mesh.get(doid)
        if mesh and mesh in drkg_dis_rev:
            return drkg_dis_rev[mesh]
    return None

# Build reverse DrugBank-ID -> node_idx map for OpenBioLink drug lookup
# (OpenBioLink drug nodes use PUBCHEM.COMPOUND* IDs; gold standard uses DrugBank IDs)
_obl_db_to_idx = {}
if 'openbilink' in kgs:
    _obl_nodes = kgs['openbilink']['nodes']
    _obl_mask  = (
        (_obl_nodes['type'] == 'Drug') &
        _obl_nodes['drugbank_id'].str.startswith('DB', na=False)
    )
    _obl_db_to_idx = dict(zip(
        _obl_nodes.loc[_obl_mask, 'drugbank_id'],
        _obl_nodes.loc[_obl_mask, 'idx'],
    ))

def find_drug_in_kg(dbid: str, drug_name: str, kg_name: str):
    nm, tm = maps[kg_name]['node_name_map'], maps[kg_name]['node_type_map']
    dt     = kg_drug_type(kg_name)
    if kg_name in ('primekg', 'biokg'):
        # Use uid_to_idx for collision-safe lookup (PrimeKG bare IDs collide across types)
        uid_map = maps[kg_name].get('uid_to_idx', {})
        # dt is a TUPLE of type labels after load_config normalisation, so
        # dt.lower() raised AttributeError here. Try every label: a KG may carry
        # drugs under more than one, and picking dt[0] would miss the rest.
        for _label in dt:
            uid_key = str(_label).lower() + '/' + dbid  # e.g. 'drug/DB15119'
            if uid_key in uid_map:
                return uid_map[uid_key]
        return maps[kg_name]['id_to_idx'].get(dbid)  # fallback for KGs without uid
    if kg_name == 'drkg':
        return find_node(dbid, nm, tm, dt)
    if kg_name == 'openbilink':
        # Drug nodes are keyed by PUBCHEM.COMPOUND* , map DrugBank ID via pre-built index
        return _obl_db_to_idx.get(dbid)
    return find_node(drug_name, nm, tm, dt)

def find_disease_in_kg(term: str, kg_name: str, disease_name_to_idx: dict):
    # Both DRKG and OpenBioLink use DOID* disease node IDs , resolve via ontology
    if kg_name in ('drkg', 'openbilink'):
        return _resolve_drkg_disease(term, disease_name_to_idx)
    t     = term.lower()
    alias = TEMPORAL_DISEASE_ALIASES.get(t, t)
    d2i   = disease_name_to_idx
    if alias in d2i:
        return d2i[alias]
    if t in d2i:
        return d2i[t]
    cands = sorted([(n, i) for n, i in d2i.items() if alias in n], key=lambda x: len(x[0]))
    if not cands:
        cands = sorted([(n, i) for n, i in d2i.items() if t in n], key=lambda x: len(x[0]))
    return cands[0][1] if cands else None

# Build disease name -> index maps per KG
disease_name_to_idx_kg = {}
for name in LOADED:
    if name in ('drkg', 'openbilink'):
        # Both route disease lookups through _resolve_drkg_disease, which uses bare CURIEs
        # like 'MESHD001234' or 'DOID10021'. Build the map with those bare keys
        # DRKG id_to_idx key = 'DiseaseMESHD001234' -> strip '' prefix -> 'MESHD001234'
        # OpenBioLink id_to_idx key = 'DOID10021' -> no '' -> unchanged
        dis_t = kg_disease_type(name)
        disease_name_to_idx_kg[name] = {
            (node_id.split('::', 1)[-1] if '::' in node_id else node_id): idx
            for node_id, idx in maps[name]['id_to_idx'].items()
            if maps[name]['node_type_map'].get(idx) in dis_t
        }
        continue
    kg_data = kgs[name]['kg']
    dst     = kg_disease_type(name)
    d2i     = {}
    for ci, cn, ct in [('x_index', 'x_name', 'x_type'), ('y_index', 'y_name', 'y_type')]:
        sub = kg_data[kg_data[ct].isin(dst)][[ci, cn]].drop_duplicates(ci)
        for _, row in sub.iterrows():
            d2i[str(row[cn]).lower()] = row[ci]
    disease_name_to_idx_kg[name] = d2i

In [21]:
# Post-cutoff approval set: tidy, ID-resolved input (preferred path)
#
# Everything above this cell is the LEGACY assembly: a hand-curated FDA CSV, a
# module of literal EMA/PMDA/TGA tuples, and an alias map translating free-text
# indication strings into Disease Ontology names. It is kept so the notebook
# still runs when the tidy file is absent, but it has three known weaknesses:
# disease lookup goes through names (an unanticipated term is silently dropped),
# the FDA half depends on a hand-curated file, and nothing validates the gold
# standard itself.
#
# When POST_CUTOFF_PATH exists it replaces all of that with one validated table
# whose identifiers are already resolved. See post_cutoff_approvals.py for the schema
# and the guards.
importlib.reload(_pg)

POST_CUTOFF_PATH = BASE / 'data/gold_standards/post_cutoff_approvals/post_cutoff_approvals.csv'
USE_TIDY_GOLD = POST_CUTOFF_PATH.exists()

DISEASE_ID_LUT = {}
if USE_TIDY_GOLD:
    PROSPECTIVE_DF, _rep = _pg.load_post_cutoff_approvals(
        POST_CUTOFF_PATH,
        do_diseases_path=BASE / 'data/gold_standards/bridges/do_diseases.csv',
        window=('2023-01-01', '2026-12-31'),
        strict=False,          # report every problem rather than dying on the first
    )
    print(_rep.render())
    NEW_INDICATION_APPROVALS = _pg.to_new_indication_approvals(PROSPECTIVE_DF)
    DISEASE_ID_LUT = _pg.disease_id_lookup(PROSPECTIVE_DF)
    print(f'\nUsing the post-cutoff approval set: {len(PROSPECTIVE_DF)} pairs, '
          f'{len(NEW_INDICATION_APPROVALS)} drugs')
else:
    PROSPECTIVE_DF = None
    print(f'Post-cutoff approval set not found at {POST_CUTOFF_PATH}')
    print('Falling back to the legacy FDA CSV + regulatory_approvals assembly '
          'above. Disease resolution is by NAME through the alias map, so terms '
          'it does not anticipate are dropped silently.')

# Identifier -> node index, per KG.
#
# The legacy resolver matches disease NAMES. Graphs key diseases by identifier
# and each uses a different one (PrimeKG MONDO, BioKG MeSH, DRKG MeSH or DOID,
# Hetionet and OpenBioLink DOID), so name matching is the weakest available
# join. Normalising both sides to a bare CURIE lets the identifier be tried
# first, with the name kept only as a fallback.

# Identifier spelling lives in post_cutoff_approvals.py so it can be unit-tested;
# the notebook only builds the per-KG index and does the lookup.
_norm_curie = _pg.norm_curie
_curie_variants = _pg.curie_variants

disease_id_to_idx_kg = {}
for _name in LOADED:
    _dst = kg_disease_type(_name)
    _m = {}
    for _node_id, _idx in maps[_name]['id_to_idx'].items():
        if maps[_name]['node_type_map'].get(_idx) not in _dst:
            continue
        _k = _norm_curie(_node_id)
        _m.setdefault(_k, _idx)
        # PrimeKG carries MONDO identifiers as bare digits; index the prefixed
        # form too so a 'MONDO:0024331' lookup can reach it.
        if _k.isdigit():
            _m.setdefault('MONDO' + _k, _idx)
            _m.setdefault('MONDO' + _k.lstrip('0'), _idx)
    disease_id_to_idx_kg[_name] = _m
print('\ndisease identifier index built per KG:',
      {k: len(v) for k, v in disease_id_to_idx_kg.items()})

def _curie_variants(v):
    """Every spelling a graph might use for one identifier, most specific first.

    Graphs disagree about whether the vocabulary prefix is part of the node id.
    Hetionet writes `Disease::DOID:8778`, BioKG writes a bare MeSH accession
    `D005317`, PrimeKG writes bare MONDO digits. Trying only the prefixed form
    matched 0 of the 92 MeSH-carrying gold rows against BioKG, which has 11,347
    disease nodes; adding the prefix-stripped form matches 90 of them.

    The bare form is only offered when it still carries a letter (`D005317`),
    never when it reduces to digits alone: `DOID:684` and a MONDO node numbered
    684 are different diseases, and a numeric match across vocabularies would
    bind an indication to the wrong node while looking like a clean hit. Bare
    numeric node ids are reached instead from the index side, where they are
    registered under their own vocabulary prefix.
    """
    k = _norm_curie(v)
    out = [k, _re.sub(r'^(MONDO|DOID|MESH)0*', r'\1', k)]
    bare = _re.sub(r'^(MONDO|DOID|MESH|UMLS|OMIM)', '', k)
    if bare and not bare.isdigit():
        out.append(bare)
    seen, ordered = set(), []
    for x in out:
        if x and x not in seen:
            seen.add(x); ordered.append(x)
    return ordered


def resolve_disease_idx(dbid, disease_term, kg_name, dis_rev):
    """Identifier first, name second.

    Returns (index, how) so the caller can report which join succeeded; a set
    resolved mostly by name is a weaker set than one resolved by identifier and
    that difference should be visible rather than inferred.
    """
    ids = DISEASE_ID_LUT.get((str(dbid), str(disease_term).lower()), {})
    idx_map = disease_id_to_idx_kg.get(kg_name, {})
    for _kind in ('mondo', 'doid', 'mesh'):
        _v = ids.get(_kind)
        if not _v:
            continue
        for _cand in _curie_variants(_v):
            if _cand in idx_map:
                return idx_map[_cand], _kind
    hit = find_disease_in_kg(disease_term, kg_name, dis_rev)
    return hit, ('name' if hit is not None else None)


post-cutoff approval set: 128/128 rows usable
  rows resolved by MONDO with no DOID available: 19

Using the post-cutoff approval set: 128 pairs, 88 drugs



disease identifier index built per KG: {'primekg': 14914, 'hetionet': 137, 'drkg': 5103, 'openbilink': 9510, 'biokg': 9490, 'matrix': 137652}


In [22]:
# Resolve FDA approval pairs per KG and build existing indication sets
existing_ind_kg, temporal_pairs_kg, pair_ids = {}, {}, {}

for name in LOADED:
    kg_data        = kgs[name]['kg']
    ntm            = maps[name]['node_type_map']
    dt, dst        = kg_drug_type(name), kg_disease_type(name)
    dis_rev        = disease_name_to_idx_kg[name]

    ind_edges = kg_data[kg_data['relation'].isin(kg_indication_rels(name))]
    # Vectorized - avoids iterrows()
    if not ind_edges.empty:
        _ntm_s = pd.Series(ntm)
        _xi = ind_edges['x_index'].to_numpy()
        _yi = ind_edges['y_index'].to_numpy()
        _xt = _ntm_s.reindex(_xi).to_numpy()
        _yt = _ntm_s.reindex(_yi).to_numpy()
        _fwd  = np.isin(_xt, dt)  & np.isin(_yt, dst)
        _rev  = np.isin(_xt, dst) & np.isin(_yt, dt)
        _keep = _fwd | _rev
        _drug = np.where(_fwd, _xi, _yi)
        _dis  = np.where(_fwd, _yi, _xi)
        exist_ind = set(zip(_drug[_keep].tolist(), _dis[_keep].tolist()))
    else:
        exist_ind = set()
    existing_ind_kg[name] = exist_ind

    pairs, unmatched_d, unmatched_dis = [], [], []
    # How each disease was joined to the graph. A set resolved mostly by name is
    # weaker than one resolved by identifier, so the split is counted and printed
    # rather than left to be assumed.
    resolved_by = {'mondo': 0, 'doid': 0, 'mesh': 0, 'name': 0}
    for dbid, (drug_name, indications) in NEW_INDICATION_APPROVALS.items():
        drug_node = find_drug_in_kg(dbid, drug_name, name)
        if drug_node is None or not graphs[name].has_node(drug_node) \
                or graphs[name].degree(drug_node) < 5:
            unmatched_d.append(dbid)
            continue
        for disease_term, approval_date, desc in indications:
            dis_idx, _how = resolve_disease_idx(dbid, disease_term, name, dis_rev)
            if _how:
                resolved_by[_how] = resolved_by.get(_how, 0) + 1
            if dis_idx is None:
                unmatched_dis.append((dbid, drug_name, disease_term))
                continue
            pairs.append((drug_node, dis_idx, approval_date, drug_name,
                          maps[name]['node_name_map'].get(dis_idx, disease_term)))
            # Keep the gold identifiers next to the node pair so the builder's per-graph
            # status (content date, distance rule) can be joined back in the next cell.
            pair_ids.setdefault(name, {})[(drug_node, dis_idx)] = (
                str(dbid), DISEASE_ID_LUT.get((str(dbid), str(disease_term).lower()), {}).get('mondo'))

    seen, unique = set(), []
    for p in pairs:
        if (p[0], p[1]) not in seen:
            seen.add((p[0], p[1]))
            unique.append(p)

    temporal_pairs_kg[name] = {
        'resolved_by':        dict(resolved_by),
        'new':                [p for p in unique if (p[0], p[1]) not in exist_ind],
        'leaked':             [p for p in unique if (p[0], p[1]) in exist_ind],
        'unmatched_drugs':    unmatched_d,
        'unmatched_diseases': unmatched_dis,
    }

n_total = len(NEW_INDICATION_APPROVALS)
print(f'{"KG":12s}  {"Drugs found":>12s}  {"New pairs":>10s}  {"Leaked":>8s}  '
      f'{"Drug miss":>10s}  {"Dis miss":>9s}')
print('-' * 70)
for name in LOADED:
    tp   = temporal_pairs_kg[name]
    n_dr = n_total - len(tp['unmatched_drugs'])
    print(f'{name:12s}  {n_dr:>5}/{n_total:<6}  {len(tp["new"]):>10,}  '
          f'{len(tp["leaked"]):>8,}  {len(tp["unmatched_drugs"]):>10,}  '
          f'{len(tp["unmatched_diseases"]):>9,}')

# Show unmatched disease terms for manual inspection
_all_unmatch_dis = set()
for name in LOADED:
    for _, _, term in temporal_pairs_kg[name]['unmatched_diseases']:
        _all_unmatch_dis.add(term)
if _all_unmatch_dis:
    print(f'\nUnresolved disease terms across all KGs ({len(_all_unmatch_dis)}):')
    for t in sorted(_all_unmatch_dis):
        print(f'  {t}')

# Identifier vs name resolution, per KG. If 'name' dominates, the identifiers in
# the gold standard are not reaching the graphs and the join is as fragile as it
# was before the tidy loader.
print()
for name in LOADED:
    rb = temporal_pairs_kg[name].get('resolved_by', {})
    tot = sum(rb.values())
    if tot:
        by_id = tot - rb.get('name', 0)
        print(f'{name:12s} resolved {by_id}/{tot} disease lookups by identifier '
              f'({rb})')


KG             Drugs found   New pairs    Leaked   Drug miss   Dis miss
----------------------------------------------------------------------
primekg          62/88              85         0          26          9
hetionet         15/88               2         0          73         20
drkg             63/88              68         0          25         25
openbilink       21/88              22         0          67          7
biokg            61/88              59         8          27         24
matrix           79/88              18        96           9          4

Unresolved disease terms across all KGs (42):
  acquired hypothalamic obesity
  adult T-cell leukemia/lymphoma
  castration-resistant prostate carcinoma
  chronic idiopathic urticaria
  chronic rhinosinusitis
  circadian rhythm sleep disorder, jet lag type
  complement 3 glomerulopathy
  differentiated thyroid carcinoma
  endometrial cancer
  enthesitis-related juvenile idiopathic arthritis
  eosinophilic granulomatosis 

In [23]:
# Checks 1 and 3 from the builder, joined to the pairs resolved above.
#
# Eligibility comes first, because the checks below are only required of
# graphs that will actually be scored on this axis.
# kg -> why it took no part, printed and checkpointed rather than left blank
prospective_skipped = {}

# Eligibility: a graph is testable on this axis only if it predates the approvals.
#
# The post-cutoff approval set runs from 2023-01-19, so a graph built after
# prospective_max_content_date already holds most of the pairs it would be
# tested on. Its coverage of the set then measures its age rather than any
# ability to predict, and the pairs that survive the per-pair date check are too
# few to estimate anything from. The rule is applied to the whole panel, before
# any score is computed, and an ineligible graph scores NaN: untestable and bad
# are different claims and must not share a number.
MAX_CONTENT_DATE = pd.Timestamp(require(params, 'prospective_max_content_date'))
with open(BASE / 'config' / 'kg_snapshots.yaml') as _fh:
    KG_SNAPSHOTS = yaml.safe_load(_fh)['snapshots']

PROSPECTIVE_ELIGIBLE = {}
print(f'Prospective eligibility (content date on or before '
      f'{MAX_CONTENT_DATE.date()}):')
for name in LOADED:
    _snap = KG_SNAPSHOTS.get(name) or {}
    _cd = _snap.get('content_date')
    _cd = pd.Timestamp(_cd) if _cd else None
    if _cd is None:
        PROSPECTIVE_ELIGIBLE[name] = False
        prospective_skipped[name] = 'no content date in config/kg_snapshots.yaml'
    elif _cd > MAX_CONTENT_DATE:
        PROSPECTIVE_ELIGIBLE[name] = False
        prospective_skipped[name] = (f'content date {_cd.date()} is after the '
                                     f'{MAX_CONTENT_DATE.date()} cutoff')
    else:
        PROSPECTIVE_ELIGIBLE[name] = True
    _v = '' if _snap.get('verified') else '  [DATE UNVERIFIED]'
    print(f'  {name:12s} {str(_cd.date()) if _cd is not None else "unknown":10s} '
          f'{"eligible" if PROSPECTIVE_ELIGIBLE[name] else "SKIPPED: " + prospective_skipped[name]}{_v}')
if not any(PROSPECTIVE_ELIGIBLE.values()):
    raise RuntimeError(
        'No graph in the panel predates the post-cutoff approval set, so this '
        'axis cannot be scored at all. Either the cutoff in config/params.yaml '
        'or the gold standard itself needs to change.')

#
# tools/build_post_cutoff_approvals.py writes post_cutoff_pair_status.csv: for every gold pair
# and every graph, whether the pair is covered, approved after the graph's content date,
# leaked as a direct edge, and related to a disease the graph already links the drug to
# (within `prospective_max_hops` is_a steps). The notebook's own resolution is the
# authority for which node a pair maps to; the status table supplies the two judgements
# the notebook cannot make from the graph alone (the content date and the distance rule),
# and the builder's own evaluable count is printed beside the notebook's as a cross-check.
STATUS_PATH   = BASE / 'data/gold_standards/post_cutoff_approvals/post_cutoff_pair_status.csv'
PROV_PATH     = BASE / 'data/gold_standards/post_cutoff_approvals/PROVENANCE.txt'
PROSPECTIVE_STRICT         = require(params, 'prospective_strict')
PROSPECTIVE_COMMON_SUPPORT = require(params, 'prospective_common_support')
MIN_NEW_PAIRS              = require(params, 'min_new_pairs')

GOLD_STANDARD_VERSION = 'unknown'
if PROV_PATH.exists():
    for _line in PROV_PATH.read_text().splitlines():
        if _line.startswith('# source_version:'):
            GOLD_STANDARD_VERSION = _line.split(':', 1)[1].strip()

PAIR_STATUS = pd.read_csv(STATUS_PATH) if STATUS_PATH.exists() else None
prospective_filter_log = {}
if PAIR_STATUS is None:
    print('post_cutoff_pair_status.csv not found: content-date and distance checks NOT applied. '
          'Run tools/build_post_cutoff_approvals.py --kgs <graphs> first.')
else:
    _eval_col = 'evaluable_strict' if PROSPECTIVE_STRICT else 'evaluable'
    _common = None
    if PROSPECTIVE_COMMON_SUPPORT:
        _wide = PAIR_STATUS.pivot_table(index=['drugbank_id', 'mondo_id'], columns='kg',
                                        values=_eval_col, aggfunc='first')
        # Only graphs that clear MIN_NEW_PAIRS on their own take part: a graph with 7
        # evaluable pairs (Hetionet) would otherwise shrink every other graph's set to
        # its own handful, and it is not scored on this axis anyway.
        _counts = PAIR_STATUS.groupby('kg')[_eval_col].sum()
        _have = [n for n in LOADED if n in _wide.columns and _counts.get(n, 0) >= MIN_NEW_PAIRS]
        _common = set(_wide.index[_wide[_have].fillna(False).astype(bool).all(axis=1)]) if _have else set()
        print(f'common support over {_have}: {len(_common)} pairs '
              f'(graphs below min_new_pairs={MIN_NEW_PAIRS} excluded: '
              f'{[n for n in LOADED if n in _wide.columns and n not in _have]})')
    _no_status = []
    for name in LOADED:
        st = PAIR_STATUS[PAIR_STATUS.kg == name].set_index(['drugbank_id', 'mondo_id'])
        if st.empty:
            print(f'{name:12s}: not in the status table (view not extracted); checks 1 and 3 skipped')
            if PROSPECTIVE_ELIGIBLE.get(name):
                _no_status.append(name)
            continue
        ids = pair_ids.get(name, {})
        before = temporal_pairs_kg[name]['new']
        kept, counts = [], {'before_content_date': 0, 'related_to_prior': 0,
                            'not_in_common_support': 0, 'not_in_status': 0}
        for p in before:
            key = ids.get((p[0], p[1]))
            if key is None or key not in st.index:
                counts['not_in_status'] += 1
                kept.append(p)                      # the notebook resolved it; the builder never saw it
                continue
            row = st.loc[key]
            if str(row['date_ok']) != 'True':
                counts['before_content_date'] += 1
                continue
            if bool(row['related_to_prior']):
                counts['related_to_prior'] += 1
                if PROSPECTIVE_STRICT:
                    continue
            if _common is not None and key not in _common:
                counts['not_in_common_support'] += 1
                continue
            kept.append(p)
        temporal_pairs_kg[name]['new'] = kept
        prospective_filter_log[name] = {'before': len(before), 'after': len(kept), **counts,
                                        'builder_' + _eval_col: int(st[_eval_col].sum())}
        print(f'{name:12s} new pairs {len(before)} -> {len(kept)}  '
              f'(before content date {counts["before_content_date"]}; related to a prior indication '
              f'{counts["related_to_prior"]}{" removed" if PROSPECTIVE_STRICT else " kept, reported"}; '
              f'outside common support {counts["not_in_common_support"]}; not in status table '
              f'{counts["not_in_status"]}); builder {_eval_col} = {int(st[_eval_col].sum())}')
    # Refuse a run where some graphs were checked and others were not.
    #
    # A graph missing from the status table keeps the pairs the content-date and
    # leakage checks would have removed, so its post-cutoff set is larger than a
    # checked graph's for a reason that has nothing to do with the graph. The
    # scores are then not comparable, and nothing downstream says so.
    if _no_status and not ALLOW_MISSING_PAIR_STATUS:
        raise RuntimeError(
            f'{len(_no_status)} of the graphs eligible for this axis are absent from '
            f'post_cutoff_pair_status.csv: {", ".join(_no_status)}. Their pairs would '
            'skip the content-date and leakage checks the other graphs passed. Rebuild '
            'the table for the whole panel:\n'
            '    python tools/build_post_cutoff_approvals.py --extract-views '
            + ' '.join(_no_status) + '\n'
            '    python tools/build_post_cutoff_approvals.py --kgs all\n'
            'or set ALLOW_MISSING_PAIR_STATUS = True in the config cell to accept an '
            'unchecked subset deliberately.')

print(f'gold standard version: {GOLD_STANDARD_VERSION}; strict={PROSPECTIVE_STRICT}; '
      f'common_support={PROSPECTIVE_COMMON_SUPPORT}')

Prospective eligibility (content date on or before 2022-12-31):
  primekg      2022-05-02 eligible
  hetionet     2017-06-08 eligible
  drkg         2020-05-04 eligible
  openbilink   2020-04-27 eligible
  biokg        2020-06-09 eligible
  matrix       2026-05-28 SKIPPED: content date 2026-05-28 is after the 2022-12-31 cutoff
primekg      new pairs 85 -> 85  (before content date 0; related to a prior indication 9 kept, reported; outside common support 0; not in status table 0); builder evaluable = 90
hetionet     new pairs 2 -> 2  (before content date 0; related to a prior indication 0 kept, reported; outside common support 0; not in status table 0); builder evaluable = 2
drkg         new pairs 68 -> 68  (before content date 0; related to a prior indication 3 kept, reported; outside common support 0; not in status table 0); builder evaluable = 60
openbilink   new pairs 22 -> 22  (before content date 0; related to a prior indication 1 kept, reported; outside common support 0; not in st

### Evaluation

In [24]:
# Prospective evaluation config and graph construction
# Renamed. This rebound MIN_PAIRS, the per-area threshold (25) used above, to a
# different quantity for a different section, so one name meant two things at
# two points in the run. The comment also said 20 while the value was 10.
# MIN_NEW_PAIRS is read from config/params.yaml (min_new_pairs) in the status cell above.

temporal_metrics_kg, temporal_graphs_kg = {}, {}
temporal_ranks_kg, baseline_all_kg, baseline_same_kg = {}, {}, {}
# kg -> {(drugbank_id, mondo_id): rank of the true pair}
prospective_ranks_by_key = {}
def _eval_pairs(pairs, n, G_eval, exclude_set, d_pool, dis_pool, nm, rng_a, rng_b):
    # Generators passed in, keyed on the KG and on which baseline this is. The
    # shared global rng made each baseline depend on how many draws the graphs
    # before it had made.
    samp = [pairs[i] for i in rng_a.choice(len(pairs), min(n, len(pairs)), replace=False)]
    neg  = generate_negatives(samp, len(samp) * NEG_RATIO, 'type-constrained',
                               d_pool, dis_pool, {}, nm, exclude_set, rng_b)
    m = compute_metrics(score_adamic_adar(G_eval, samp + neg),
                        [1]*len(samp) + [0]*len(neg))
    m['n_sampled'] = len(samp)
    m['n_neg'] = len(neg)
    return m

In [25]:
# Prospective evaluation , main loop (with bootstrap CIs)
# Populates temporal_metrics_kg, temporal_ranks_kg, baseline_all_kg, baseline_same_kg

# Per-KG bootstrap streams. One shared BOOT_RNG consumed in loop order made
# every interval depend on how many KGs were bootstrapped before it.

for name in LOADED:
    tp_info   = temporal_pairs_kg[name]
    new_pairs = [(p[0], p[1]) for p in tp_info['new']]  # 2-tuples for scoring

    if not PROSPECTIVE_ELIGIBLE[name]:
        print(f'{name}: skipped, {prospective_skipped[name]}')
        continue

    if len(new_pairs) < MIN_NEW_PAIRS:
        print(f'{name}: skipped, only {len(new_pairs)} new pair(s) '
              f'< MIN_NEW_PAIRS={MIN_NEW_PAIRS}')
        prospective_skipped[name] = (f'{len(new_pairs)} evaluable pair(s), below '
                                     f'min_new_pairs={MIN_NEW_PAIRS}')
        continue

    G_eval   = graphs[name]
    ents     = kg_entity_sets[name]
    d_pool   = ents['drug']
    dis_pool = ents['disease']
    nm       = maps[name]['node_name_map']
    pos_set  = set(kg_pos_pairs[name])
    exclude  = pos_set | set(new_pairs)  # exclude both known and new pairs from negatives

    n_eval   = len(new_pairs)

    # Evaluate new (prospective) pairs
    samp_idx = _kg_rng(name, 'prospective-sample').choice(
        len(new_pairs), min(n_eval, len(new_pairs)), replace=False)
    sampled  = [new_pairs[i] for i in samp_idx]
    neg      = generate_negatives(sampled, len(sampled) * NEG_RATIO, 'type-constrained',
                                   d_pool, dis_pool, {}, nm, exclude,
                                   _kg_rng(name, 'prospective-neg'))
    all_scores = score_adamic_adar(G_eval, sampled + neg)
    labels     = [1] * len(sampled) + [0] * len(neg)

    m = compute_metrics(all_scores, labels)

    # Bootstrap 95% CIs for AUROC and AUPRC
    scores_arr = np.array(all_scores, dtype=float)
    labels_arr = np.array(labels, dtype=int)
    _, auroc_lo, auroc_hi = bootstrap_auroc_ci(
        scores_arr, labels_arr, n_boot=N_BOOTSTRAP,
        rng_=_kg_rng(name, 'prospective-boot-auroc'))
    _, auprc_lo, auprc_hi = bootstrap_auprc_ci(
        scores_arr, labels_arr, n_boot=N_BOOTSTRAP,
        rng_=_kg_rng(name, 'prospective-boot-auprc'))
    m['auroc_ci_lo'] = auroc_lo
    m['auroc_ci_hi'] = auroc_hi
    m['auprc_ci_lo'] = auprc_lo
    m['auprc_ci_hi'] = auprc_hi

    # Disease-clustered intervals.
    #
    # Several prospective pairs share a disease, and every positive carries its
    # own generated negatives, so rows within a disease share a candidate pool,
    # a neighbourhood and a difficulty. The row bootstrap above treats them as
    # independent and is therefore too narrow: on synthetic data with realistic
    # disease-level difficulty the clustered interval is ~1.4x wider. Both are
    # recorded so the difference is visible; report the clustered one.
    cluster_ids = np.array([p[1] for p in sampled] + [n[1] for n in neg])
    _, c_auroc_lo, c_auroc_hi = bootstrap_auroc_ci_clustered(
        scores_arr, labels_arr, cluster_ids, n_boot=N_BOOTSTRAP,
        rng_=_kg_rng(name, 'prospective-boot-auroc-clustered'))
    _, c_auprc_lo, c_auprc_hi = bootstrap_auprc_ci_clustered(
        scores_arr, labels_arr, cluster_ids, n_boot=N_BOOTSTRAP,
        rng_=_kg_rng(name, 'prospective-boot-auprc-clustered'))
    m['auroc_ci_lo_clustered'] = c_auroc_lo
    m['auroc_ci_hi_clustered'] = c_auroc_hi
    m['auprc_ci_lo_clustered'] = c_auprc_lo
    m['auprc_ci_hi_clustered'] = c_auprc_hi
    m['n_disease_clusters']    = int(len(np.unique(cluster_ids)))

    # Drug-clustered intervals as well. A few drugs contribute many pairs (one PD-1 inhibitor
    # brings eight new tumour types), so rows also share a drug neighbourhood and a difficulty.
    # Both clusterings are recorded; report the wider of the two.
    drug_cluster_ids = np.array([p[0] for p in sampled] + [n[0] for n in neg])
    _, d_auroc_lo, d_auroc_hi = bootstrap_auroc_ci_clustered(
        scores_arr, labels_arr, drug_cluster_ids, n_boot=N_BOOTSTRAP,
        rng_=_kg_rng(name, 'prospective-boot-auroc-clustered-drug'))
    m['auroc_ci_lo_clustered_drug'] = d_auroc_lo
    m['auroc_ci_hi_clustered_drug'] = d_auroc_hi
    m['n_drug_clusters']            = int(len(np.unique(drug_cluster_ids)))

    # Extra Hits@50 / Hits@100
    pos_sc = all_scores[np.array(labels) == 1]
    hits50, hits100, raw_ranks = [], [], []
    for ps in pos_sc:
        rank = int(np.sum(all_scores >= ps))
        hits50.append(1 if rank <= 50 else 0)
        hits100.append(1 if rank <= 100 else 0)
        raw_ranks.append(rank)

    m['hits@50']     = float(np.mean(hits50))
    m['hits@100']    = float(np.mean(hits100))
    # The rank above is computed against the SCORED candidate set, the sampled
    # positives plus their negatives, not against every disease in the graph.
    # n_candidates is that denominator. n_diseases is kept because other cells
    # report it, but it is not the base of these ranks: the percentile figure
    # divided by it and so reported ranks out of ~120 as a fraction of ~9,500.
    m['n_candidates'] = int(len(labels))
    m['n_diseases']   = len(dis_pool)
    m['n_new_pairs']  = len(new_pairs)
    m['n_sampled']    = len(sampled)
    m['n_neg']        = len(neg)
    # Hits@K is only meaningful while K is small relative to the candidate set.
    # With NEG_RATIO negatives per positive that set is (1+NEG_RATIO)*n_sampled,
    # so Hits@100 can sit near 1 by construction on a small post-cutoff set.
    if len(labels) <= 100:
        print(f'  NOTE {name}: only {len(labels)} candidates scored, so '
              f'Hits@100 saturates and Hits@50 is near the pool size too')

    # Ranks keyed on (drugbank_id, mondo_id) rather than on this graph's node
    # ids, so two graphs can be compared on the pairs both could evaluate.
    # Without a shared key the only comparison available is between whole-set
    # summaries computed on different samples.
    _ids = pair_ids.get(name, {})
    prospective_ranks_by_key[name] = {
        _ids[(p[0], p[1])]: int(r)
        for p, r in zip(sampled, raw_ranks) if (p[0], p[1]) in _ids}
    _unkeyed = len(sampled) - len(prospective_ranks_by_key[name])
    if _unkeyed:
        print(f'  NOTE {name}: {_unkeyed} evaluated pair(s) carry no gold key, '
              'so they take no part in the pairwise comparison')

    temporal_metrics_kg[name] = m
    temporal_ranks_kg[name]   = np.array(raw_ranks)
    temporal_graphs_kg[name]  = G_eval

    # Baseline (all existing indication pairs)
    baseline_all_kg[name] = _eval_pairs(
        kg_pos_pairs[name], n_eval, G_eval, set(), d_pool, dis_pool, nm,
        _kg_rng(name, 'baseline-all-sample'), _kg_rng(name, 'baseline-all-neg'))

    # Baseline (same drugs as post-cutoff set)
    drug_nodes   = {p[0] for p in sampled}
    same_d_pairs = [(d, dis) for d, dis in kg_pos_pairs[name] if d in drug_nodes]
    if len(same_d_pairs) >= 1:
        baseline_same_kg[name] = _eval_pairs(
            same_d_pairs, n_eval, G_eval, set(), d_pool, dis_pool, nm,
            _kg_rng(name, 'baseline-same-sample'),
            _kg_rng(name, 'baseline-same-neg'))
    else:
        baseline_same_kg[name] = {k: float('nan')
                                   for k in ['auroc', 'auprc', 'mrr', 'hits@10']}

    auroc_ci_str = f'[{auroc_lo:.3f}, {auroc_hi:.3f}]' if not np.isnan(auroc_lo) else 'N/A'
    auprc_ci_str = f'[{auprc_lo:.3f}, {auprc_hi:.3f}]' if not np.isnan(auprc_lo) else 'N/A'
    cl_str = (f'[{c_auroc_lo:.3f}, {c_auroc_hi:.3f}]'
              if not np.isnan(c_auroc_lo) else 'N/A')
    print(f'{name}: {len(new_pairs)} new pairs over '
          f'{m["n_disease_clusters"]} disease(s)  '
          f'AUROC={m["auroc"]:.4f}  row CI {auroc_ci_str}  '
          f'clustered CI {cl_str}  '
          f'AUPRC={m["auprc"]:.4f} {auprc_ci_str}  '
          f'Hits@10={m["hits@10"]:.1%}')
    if m['n_disease_clusters'] < 10:
        print(f'  NOTE {name}: only {m["n_disease_clusters"]} distinct diseases, '
              f'so the clustered interval rests on very few independent units')

primekg: 85 new pairs over 487 disease(s)  AUROC=0.7764  row CI [0.723, 0.829]  clustered CI [0.717, 0.832]  AUPRC=0.5725 [0.476, 0.671]  Hits@10=10.6%
hetionet: skipped, only 2 new pair(s) < MIN_NEW_PAIRS=10


drkg: 68 new pairs over 375 disease(s)  AUROC=0.8679  row CI [0.816, 0.916]  clustered CI [0.811, 0.919]  AUPRC=0.7056 [0.605, 0.806]  Hits@10=13.2%


openbilink: 22 new pairs over 130 disease(s)  AUROC=0.6769  row CI [0.581, 0.784]  clustered CI [0.571, 0.780]  AUPRC=0.4358 [0.282, 0.621]  Hits@10=36.4%


biokg: 59 new pairs over 338 disease(s)  AUROC=0.8306  row CI [0.774, 0.889]  clustered CI [0.764, 0.892]  AUPRC=0.5731 [0.465, 0.726]  Hits@10=11.9%
matrix: skipped, content date 2026-05-28 is after the 2022-12-31 cutoff


In [26]:
# Prospective evaluation summary
def _fmt(v, fmt):
    return format(v, fmt) if not (isinstance(v, float) and np.isnan(v)) else 'N/A'

print(f'{"KG":12s}  {"Base (all)":>10s}  {"Base (same)":>11s}  {"New pairs":>10s}  '
      f'{"Gap%":>6s}  {"Hits@10":>8s}  {"MRR":>7s}')
print('-' * 75)
for name in LOADED:
    m   = temporal_metrics_kg.get(name, {})
    ba  = baseline_all_kg.get(name,  {}).get('auroc', np.nan)
    bs  = baseline_same_kg.get(name, {}).get('auroc', np.nan)
    fut = m.get('auroc', np.nan)
    gap = (ba - fut) / ba * 100 if not (np.isnan(ba) or np.isnan(fut) or ba == 0) else np.nan
    print(f'{name:12s}  {_fmt(ba, ".4f"):>10s}  {_fmt(bs, ".4f"):>11s}  '
          f'{_fmt(fut, ".4f"):>10s}  {_fmt(gap, ".1f"):>6s}  '
          f'{_fmt(m.get("hits@10", np.nan), ".1%"):>8s}  '
          f'{_fmt(m.get("mrr", np.nan), ".4f"):>7s}')

KG            Base (all)  Base (same)   New pairs    Gap%   Hits@10      MRR
---------------------------------------------------------------------------
primekg           0.9705       0.9914      0.7764    20.0     10.6%   0.0504
hetionet             N/A          N/A         N/A     N/A       N/A      N/A
drkg              0.9361       0.9666      0.8679     7.3     13.2%   0.0634
openbilink        0.7045       0.7702      0.6769     3.9     36.4%   0.1220
biokg             0.8980       0.9646      0.8306     7.5     11.9%   0.0461
matrix               N/A          N/A         N/A     N/A       N/A      N/A


### Coverage, retrieval and pairwise comparison

Three numbers, because no one of them is honest on its own. A graph can only be
scored on the pairs it can express, and the graphs differ by a factor of sixty in
how many that is, so a conditional score on a small evaluable subset is not
comparable with one computed on most of the gold set. The subsets are also not
missing at random: an older graph holds the better-studied drugs, which are the
easier ones.

| | question | denominator |
|---|---|---|
| **Coverage** | how often can the graph answer? | evaluable / all gold pairs |
| **Hits@K (all)** | can it deliver a future approval at all? | all gold pairs, inexpressible counted as a miss |
| **AUROC** | when it answers, is the ranking good? | its own evaluable set |

Hits@K over the whole gold set needs no imputation: its denominator is the gold
pairs, so a pair the graph cannot rank is a genuine failure to retrieve. AUROC
cannot be treated that way, because it is defined over a ranked candidate list
and a pair with no nodes has no list; inserting a zero there would assert that
the true positive ranked last and would push the graph below AUROC's 0.5 chance
floor. The scored sub-metric stays the conditional AUROC, since coverage is
already scored in dimension 01 and charging it again here would count the same
deficiency twice in Fig 1.

The pairwise table is the complete-case check on all of that: two graphs
compared only on the pairs both can evaluate. The panel-wide intersection is
useless here (one low-coverage graph collapses it to almost nothing), but
pairwise intersections stay large enough to say whether a graph really ranks
better or merely drew an easier sample.

In [27]:
# Coverage, full-denominator retrieval, and the pairwise complete-case comparison.
#
# N_GOLD is the whole post-cutoff set, not the evaluable subset: it is the
# denominator that makes "could not express this approval" count as the miss it
# is, rather than disappearing from the arithmetic.
N_GOLD = len(PROSPECTIVE_DF) if USE_TIDY_GOLD else sum(
    len(v) for v in NEW_INDICATION_APPROVALS.values())
HITS_K = require(params, 'prospective_hits_k')

prospective_coverage_df = prospective_summary(
    prospective_ranks_by_key, n_gold=N_GOLD, k=HITS_K)
out.table(prospective_coverage_df, 'prospective_coverage')
print(f'Post-cutoff approval set: {N_GOLD} pairs. Hits@{HITS_K} "all" counts a pair the '
      'graph cannot express as a miss; "evaluable" conditions on the subset it can.')
display(prospective_coverage_df.set_index('KG'))

# Pairwise, on the pairs both graphs can evaluate.
prospective_pairwise_df = pairwise_agreement(
    prospective_ranks_by_key, min_shared=MIN_NEW_PAIRS)
out.table(prospective_pairwise_df, 'prospective_pairwise')
if prospective_skipped:
    print()
    print('Graphs taking no part in the prospective axis, and why. Each scores NaN '
          'rather than 0: a graph that cannot be tested has not failed the test.')
    for _n, _why in prospective_skipped.items():
        print(f'  {_n:12s} {_why}')

if prospective_pairwise_df.empty:
    print('No pair of graphs shares an evaluable pair; nothing to compare.')
else:
    display(prospective_pairwise_df.set_index(['kg_a', 'kg_b']))
    _weak = prospective_pairwise_df[~prospective_pairwise_df.comparable]
    if len(_weak):
        print(f'{len(_weak)} of {len(prospective_pairwise_df)} comparisons rest on '
              f'fewer than {MIN_NEW_PAIRS} shared pairs and are marked not comparable.')
    print('a_win_rate is the share of DECIDED shared pairs where kg_a ranked the true '
          'approval better; 0.5 means the two order them about equally well. Ties are '
          'excluded from the rate and reported separately.')

Post-cutoff approval set: 128 pairs. Hits@10 "all" counts a pair the graph cannot express as a miss; "evaluable" conditions on the subset it can.


,n_evaluable,n_gold,coverage,hits@10_all,hits@10_evaluable
KG,,,,,
primekg,85,128,0.6641,0.0703,0.1059
drkg,68,128,0.5312,0.0703,0.1324
openbilink,22,128,0.1719,0.0625,0.3636
biokg,59,128,0.4609,0.0547,0.1186



Graphs taking no part in the prospective axis, and why. Each scores NaN rather than 0: a graph that cannot be tested has not failed the test.
  matrix       content date 2026-05-28 is after the 2022-12-31 cutoff
  hetionet     2 evaluable pair(s), below min_new_pairs=10


n_shared  a_better  b_better  ties  a_win_rate  \
kg_a       kg_b                                                         
drkg       primekg           60        26        32     2      0.4483   
biokg      drkg              56        37        19     0      0.6607   
           primekg           53        29        24     0      0.5472   
openbilink primekg           20        11         9     0      0.5500   
drkg       openbilink        19        11         8     0      0.5789   
biokg      openbilink        11         6         5     0      0.5455   

                       comparable  
kg_a       kg_b                    
drkg       primekg           True  
biokg      drkg              True  
           primekg           True  
openbilink primekg           True  
drkg       openbilink        True  
biokg      openbilink        True

a_win_rate is the share of DECIDED shared pairs where kg_a ranked the true approval better; 0.5 means the two order them about equally well. Ties are excluded from the rate and reported separately.


### Visualisation

In [28]:
out.figure(fig07_prospective_auroc(baseline_all_kg, baseline_same_kg,
                                  temporal_metrics_kg),
           '07d_prospective_auroc');


  saved 07d_prospective_auroc.pdf and .png (600 dpi)


In [29]:
out.figure(fig07_prospective_hits(temporal_metrics_kg,
                                 [n for n in LOADED if area_results_kg.get(n)],
                                 KG_COLORS),
           '07e_prospective_hits');


  saved 07e_prospective_hits.pdf and .png (600 dpi)


In [30]:
# Build percentile rank data for the rank distribution plot
pct_data = {}
all_pct = []
for name in temporal_ranks_kg:
    ranks = temporal_ranks_kg[name]
    # Divide by the number of candidates the rank was computed against, not by
    # the size of the disease vocabulary.
    n_total = temporal_metrics_kg[name].get('n_candidates', len(ranks))
    pct = 100.0 * ranks / max(n_total, 1)
    pct_data[name] = pct
    all_pct.extend(pct.tolist())

print(f'Percentile data built for {len(pct_data)} KGs: {list(pct_data.keys())}')


Percentile data built for 4 KGs: ['primekg', 'drkg', 'openbilink', 'biokg']


In [31]:
out.figure(fig07_rank_distribution(pct_data, KG_COLORS), '07f_rank_distribution');


  saved 07f_rank_distribution.pdf and .png (600 dpi)


### Network proximity validation

Chandak et al. (2023) validated PrimeKG by showing that newly approved drugs sit
significantly closer to their indicated diseases than to random diseases in the
graph. We replicate this analysis across all six KGs using the FDA novel drug
approval set.

**Method.** For each drug-disease pair found in a KG, we compute the unweighted
shortest-path distance, then run a permutation test (1,000 random disease nodes)
to assess whether the true indication is significantly closer than expected by
chance. P-values are Bonferroni-corrected per KG.

In [32]:
random.seed(42)
np.random.seed(42)

N_PERM = 1000

# Entity-type labels per KG (disease node type)
_DIS_TYPE = {
    'primekg': 'disease', 'hetionet': 'Disease', 'drkg': 'Disease',
    'openbilink': 'Disease', 'biokg': 'Disease',
    'matrix': 'disease',
}

all_proximity = {}

for kg_name in LOADED:
    t0 = time.time()
    G = graphs[kg_name]
    nodes_df = kgs[kg_name]['nodes']

    # All disease node indices (for permutation pool)
    dis_type = _DIS_TYPE.get(kg_name, 'Disease')
    all_dis_idxs = nodes_df[nodes_df['type'] == dis_type]['idx'].tolist()

    # Resolve FDA drug-disease pairs to graph node indices
    pairs = []
    seen  = set()
    for _, row in _fda.iterrows():
        db_id      = str(row['DrugBank_ID']).strip()
        indication = str(row['Indication']).strip()
        drug_name  = str(row['Active Ingredient']).strip()

        drug_idx = find_drug_in_kg(db_id, drug_name, kg_name)
        if drug_idx is None:
            continue

        dis_idx = find_disease_in_kg(indication, kg_name,
                                     disease_name_to_idx_kg[kg_name])
        if dis_idx is None:
            continue

        key = (drug_idx, dis_idx)
        if key in seen:  # avoid duplicate pairs
            continue
        seen.add(key)

        pairs.append({
            'drug': drug_name, 'db_id': db_id, 'indication': indication,
            'drug_idx': drug_idx, 'dis_idx': dis_idx,
        })

    print(f'\n{"=" * 70}')
    print(f'{kg_name.upper()}: {len(pairs)} testable drug\u2013disease pairs')
    print(f'{"=" * 70}')

    results = []
    for i, pair in enumerate(pairs):
        # Single-source BFS O(V+E) once per drug/ Scale-aware bound BFS depth for huge graphs. Drug-disease distances
        # >5 hops are noise for permutation tests anyway.
        _bfs_cutoff = 6 if G.number_of_nodes() > 1_000_000 else None
        dists = dict(nx.single_source_shortest_path_length(
            G, pair['drug_idx'], cutoff=_bfs_cutoff))

        obs = dists.get(pair['dis_idx'])
        if obs is None:
            print(f"  {pair['drug']:35s}  DISCONNECTED")
            continue

        # Permutation pool reachable diseases only
        pool = [d for d in all_dis_idxs if d in dists and d != pair['dis_idx']]
        if len(pool) < 100:
            print(f"  {pair['drug']:35s}  TOO FEW REACHABLE DISEASES ({len(pool)})")
            continue

        sample     = random.sample(pool, min(N_PERM, len(pool)))
        rand_dists = [dists[d] for d in sample]

        raw_p = sum(1 for d in rand_dists if d <= obs) / len(rand_dists)

        results.append({
            **pair,
            'obs_dist':        obs,
            'random_mean':     np.mean(rand_dists),
            'random_95ci_low': np.percentile(rand_dists, 2.5),
            'random_95ci_high':np.percentile(rand_dists, 97.5),
            'raw_p':           raw_p,
            'n_random':        len(rand_dists),
        })

        if (i + 1) % 25 == 0:
            print(f'  [{i+1}/{len(pairs)}] {time.time() - t0:.0f}s elapsed ...')

    # Bonferroni correction
    n_tested = len(results)
    for r in results:
        r['bonf_p']      = min(r['raw_p'] * n_tested, 1.0)
        r['significant'] = r['bonf_p'] <= 0.05

    elapsed = time.time() - t0
    n_sig   = sum(1 for r in results if r['significant'])
    print(f'\n  {n_tested} tested, {n_sig} significant '
          f'(Bonferroni \u2264 0.05)  [{elapsed:.1f}s]')

    all_proximity[kg_name] = results


PRIMEKG: 49 testable drug–disease pairs


  [25/49] 17s elapsed ...



  49 tested, 3 significant (Bonferroni ≤ 0.05)  [32.7s]



HETIONET: 0 testable drug–disease pairs

  0 tested, 0 significant (Bonferroni ≤ 0.05)  [0.6s]



DRKG: 54 testable drug–disease pairs


  [25/54] 22s elapsed ...


... (7 outputs left out)

  [50/122] 1449s elapsed ...


  [75/122] 2159s elapsed ...


  [100/122] 2854s elapsed ...



  122 tested, 66 significant (Bonferroni ≤ 0.05)  [3444.4s]


In [33]:
# Free per-KG edge DataFrames
for _name in list(kgs):
    kgs[_name].pop('kg', None)
print('Released kgs[*][\'kg\'] DataFrames.')

Released kgs[*]['kg'] DataFrames.


In [34]:
# Network proximity results tables
proximity_tables = {}

for kg_name in LOADED:
    results = all_proximity.get(kg_name, [])
    ok = [r for r in results if 'obs_dist' in r]
    if not ok:
        print(f'\n{kg_name}: no testable pairs')
        continue

    rows = []
    for r in sorted(ok, key=lambda x: (x['obs_dist'], x['bonf_p'])):
        rows.append({
            'Drug':        r['drug'],
            'Disease':     r['indication'],
            'Obs dist':    r['obs_dist'],
            'Rand mean':   round(r['random_mean'], 2),
            '95% CI':      f"[{r['random_95ci_low']:.1f}, {r['random_95ci_high']:.1f}]",
            'Raw P':       round(r['raw_p'], 4),
            'Bonf P':      round(r['bonf_p'], 4),
            'Sig':         '\u2713' if r['significant'] else '\u2717',
        })

    tbl  = pd.DataFrame(rows)
    proximity_tables[kg_name] = tbl
    n_sig = sum(1 for r in ok if r.get('significant'))

    print(f'\n{"=" * 90}')
    print(f'{kg_name.upper()}: {len(ok)} pairs tested, {n_sig} significant '
          '(Bonferroni \u2264 0.05)')
    print(f'{"=" * 90}')
    display(tbl)

# Cross-KG summary
print(f'\n{"=" * 90}')
print('CROSS-KG NETWORK PROXIMITY SUMMARY')
print(f'{"=" * 90}')

summary_rows = []
for kg_name in LOADED:
    res = all_proximity.get(kg_name, [])
    ok  = [r for r in res if 'obs_dist' in r]
    n_sig = sum(1 for r in ok if r.get('significant'))
    summary_rows.append({
        'KG':                   kg_name,
        'Pairs tested':         len(ok),
        'Significant':          n_sig,
        '% significant':        f'{100 * n_sig / len(ok):.0f}%' if ok else 'N/A',
        'Median obs dist':      int(np.median([r['obs_dist'] for r in ok])) if ok else None,
        'Median random mean':   round(np.median([r['random_mean'] for r in ok]), 2) if ok else None,
    })

display(pd.DataFrame(summary_rows).set_index('KG'))


PRIMEKG: 49 pairs tested, 3 significant (Bonferroni ≤ 0.05)


,Drug,Disease,Obs dist,Rand mean,95% CI,Raw P,Bonf P,Sig
0,gepirone,Major depressive disorder,1,3.49,"[2.0, 5.0]",0.001,0.049,✓
1,inclisiran,Familial hypercholesterolemia / ASCVD,2,4.63,"[4.0, 6.0]",0.000,0.000,✓
2,ganaxolone,CDKL5 deficiency disorder seizures,2,4.32,"[3.0, 6.0]",0.000,0.000,✓
3,oomidenepag isopropyl,Open-angle glaucoma / ocular hypertension,2,3.99,"[3.0, 6.0]",0.003,0.147,✗
4,mavorixafor,WHIM syndrome,2,4.38,"[3.0, 6.0]",0.003,0.147,✗
5,quizartinib,Newly diagnosed AML (FLT3-ITD),2,4.07,"[3.0, 6.0]",0.007,0.343,✗
6,elacestrant,ER+/HER2- ESR1-mutated advanced breast cancer,2,4.10,"[3.0, 6.0]",0.010,0.490,✗
7,sparsentan,IgA nephropathy,2,4.14,"[3.0, 6.0]",0.011,0.539,✗
8,lebrikizumab-lbkz,Moderate-to-severe atopic dermatitis,2,3.75,"[3.0, 5.0]",0.013,0.637,✗
9,adagrasib,KRAS G12C-mutated NSCLC,2,4.24,"[3.0, 6.0]",0.017,0.833,✗



hetionet: no testable pairs

DRKG: 53 pairs tested, 1 significant (Bonferroni ≤ 0.05)


,Drug,Disease,Obs dist,Rand mean,95% CI,Raw P,Bonf P,Sig
0,fexinidazole,Human African trypanosomiasis,1,3.71,"[3.0, 5.0]",0.000,0.000,✓
1,momelotinib,Myelofibrosis with anemia,1,3.09,"[2.0, 4.0]",0.001,0.053,✗
2,adagrasib,KRAS G12C-mutated NSCLC,2,3.63,"[3.0, 4.0]",0.010,0.530,✗
3,tirzepatide,Type 2 diabetes mellitus,2,2.80,"[2.0, 4.0]",0.376,1.000,✗
4,tapinarof,Plaque psoriasis,2,3.12,"[2.0, 4.0]",0.104,1.000,✗
5,terlipressin,Hepatorenal syndrome,2,2.83,"[2.0, 4.0]",0.340,1.000,✗
6,tremelimumab,Unresectable hepatocellular carcinoma,2,3.21,"[2.0, 4.0]",0.056,1.000,✗
7,mirvetuximab soravtansine-gynx,Recurrent platinum-resistant ovarian cancer,2,2.86,"[2.0, 4.0]",0.314,1.000,✗
8,ublituximab-xiiy,Relapsing multiple sclerosis,2,3.20,"[2.0, 4.0]",0.043,1.000,✗
9,sotagliflozin,Heart failure,2,2.72,"[2.0, 4.0]",0.401,1.000,✗



OPENBILINK: 13 pairs tested, 2 significant (Bonferroni ≤ 0.05)


,Drug,Disease,Obs dist,Rand mean,95% CI,Raw P,Bonf P,Sig
0,gepirone,Major depressive disorder,1,4.17,"[3.0, 6.0]",0.000,0.000,✓
1,mavorixafor,WHIM syndrome,2,4.03,"[3.0, 6.0]",0.000,0.000,✓
2,sotagliflozin,Heart failure,2,4.12,"[3.0, 7.0]",0.005,0.065,✗
3,quizartinib,Newly diagnosed AML (FLT3-ITD),2,3.79,"[3.0, 6.0]",0.016,0.208,✗
4,capivasertib,Breast cancer (AKT pathway),2,3.81,"[2.0, 6.0]",0.026,0.338,✗
5,seladelpar,Primary biliary cholangitis,3,4.53,"[3.0, 7.0]",0.039,0.507,✗
6,terlipressin,Hepatorenal syndrome,3,3.69,"[2.0, 6.0]",0.501,1.000,✗
7,bexagliflozin,Type 2 diabetes mellitus,3,4.38,"[3.0, 7.0]",0.193,1.000,✗
8,momelotinib,Myelofibrosis with anemia,3,3.84,"[3.0, 6.0]",0.425,1.000,✗
9,mirdametinib,Neurofibromatosis type 1 (plexiform neurofibro...,3,3.74,"[3.0, 6.0]",0.485,1.000,✗


... (2 outputs left out)


MATRIX: 122 pairs tested, 66 significant (Bonferroni ≤ 0.05)


,Drug,Disease,Obs dist,Rand mean,95% CI,Raw P,Bonf P,Sig
0,ibrexafungerp,Vulvovaginal candidiasis,1,3.00,"[2.0, 5.0]",0.000,0.0,✓
1,fexinidazole,Human African trypanosomiasis,1,2.96,"[2.0, 5.0]",0.000,0.0,✓
2,odevixibat,Pruritus (PFIC),1,3.25,"[2.0, 5.0]",0.000,0.0,✓
3,belzutifan,Von Hippel-Lindau disease,1,3.05,"[2.0, 5.0]",0.000,0.0,✓
4,difelikefalin,Pruritus in chronic kidney disease,1,3.11,"[2.0, 5.0]",0.000,0.0,✓
...,...,...,...,...,...,...,...,...
117,aficamten,Symptomatic obstructive HCM,3,4.13,"[3.0, 6.0]",0.192,1.0,✗
118,teclistamab-cqyv,Relapsed/refractory multiple myeloma,4,4.02,"[3.0, 6.0]",0.764,1.0,✗
119,olezarsen,Familial chylomicronemia syndrome,4,4.58,"[4.0, 6.0]",0.500,1.0,✗
120,fitusiran,Hemophilia A or B,4,4.67,"[4.0, 6.0]",0.451,1.0,✗



CROSS-KG NETWORK PROXIMITY SUMMARY


,Pairs tested,Significant,% significant,Median obs dist,Median random mean
KG,,,,,
primekg,49,3,6%,2.0,3.76
hetionet,0,0,N/A,NaN,NaN
drkg,53,1,2%,3.0,3.23
openbilink,13,2,15%,3.0,4.03
biokg,20,2,10%,3.0,3.59
matrix,122,66,54%,1.0,3.11


In [35]:
out.figure(fig07_proximity(all_proximity), '07g_network_proximity');


  saved 07g_network_proximity.pdf and .png (600 dpi)


### Checkpoint

## Dimension score

The **Generalisation** dimension score aggregates three complementary axes, each
measured as an AUROC in $[0, 1]$:

| Sub-score | Metric | Rationale |
|-----------|--------|-----------|
| **Data-sparse generalisation** | AUROC on Q1 (sparse-tier) diseases | Measures performance on rare / under-represented diseases, the hardest generalisation setting |
| **Cross-domain generalisation** | Mean AUROC across therapeutic areas | Captures whether predictive power is consistent across disease domains |
| **Prospective generalisation** | AUROC on post-cutoff FDA approvals | Tests prospective validity, whether the KG predicts indications approved after its data freeze |

$$
S_{\text{gen}}(g) \;=\; \frac{1}{3}\!\left(
   \text{AUROC}_{\text{Q1}}(g)
   \;+\; \overline{\text{AUROC}}_{\text{area}}(g)
   \;+\; \text{AUROC}_{\text{prospective}}(g)
\right)
$$

All sub-scores are saved in `sub_scores[kg]` for the summary heatmap in notebook 00.

In [36]:
# Checkpoint and tables for downstream notebooks.
#
# Written through `out`, as every other dimension does, so the checkpoint stem
# matches the notebook directory and the run reports what it produced. This
# cell previously pickled by hand AND rebound the name `out` to the checkpoint
# path, which would now clobber the Outputs handle for anything after it.
CKPT_DIR = BASE / 'results' / 'checkpoints'
CKPT_DIR.mkdir(parents=True, exist_ok=True)

def _scalars(metrics: dict) -> dict:
    '''Strip non-scalar values from a metrics dict.'''
    return {k: float(v) for k, v in metrics.items()
            if isinstance(v, (int, float)) and not hasattr(v, '__len__')}

def _safe_mean(vals):
    '''Mean of non-NaN values, or NaN if empty.'''
    clean = [v for v in vals if not (isinstance(v, float) and np.isnan(v))]
    return np.mean(clean) if clean else np.nan

# Sub-scores (one per generalisation axis)
RESCALE_AUROC = True   # match notebook 06; see the note at sub_scores below


def _rescale(a):
    '''AUROC to a 0-at-chance scale, as notebook 06 does for link prediction.'''
    if a is None or (isinstance(a, float) and np.isnan(a)):
        return np.nan
    return round(float(np.clip((a - 0.5) / 0.5, 0.0, 1.0)), 4) if RESCALE_AUROC \
        else round(float(a), 4)


sub_scores, sub_scores_raw = {}, {}
# Coverage and retrieval, keyed by KG, from the section above.
_cov = prospective_coverage_df.set_index('KG').to_dict('index')
for name in LOADED:
    # 1. Data-sparse Q1 (sparse tier) AUROC
    q1_auroc = tier_results_kg.get(name, {}).get('Q1: Sparse', {}).get('auroc', np.nan)

    # 2. Cross-domain mean AUROC across therapeutic areas
    area_aurocs = [
        m.get('auroc', np.nan)
        for m in area_results_kg.get(name, {}).values()
        if not np.isnan(m.get('auroc', np.nan))
    ]
    cd_auroc = np.mean(area_aurocs) if area_aurocs else np.nan

    # 3. Prospective AUROC on post-cutoff FDA approval pairs
    # NaN if KG had fewer than MIN_PAIRS evaluable pairs (unreliable estimate)
    tmp_auroc = temporal_metrics_kg.get(name, {}).get('auroc', np.nan)

    # All three axes are AUROCs, with a chance floor of 0.5. Notebook 06
    # rescales its AUROC sub-metric to (AUROC - 0.5) / 0.5 so that it shares a
    # floor of 0 with its proportion-valued sub-metrics. Left raw here, the two
    # dimensions sit on different scales: a graph at chance scores 0.0 on task
    # performance and 0.5 on generalisation, and notebook 00 puts both in one
    # heatmap and one radar. Rescaled the same way, the two are comparable and
    # the floor means the same thing everywhere.
    #
    # The raw AUROCs are kept in the checkpoint, because they are the numbers
    # comparable with the literature and with the figures in this notebook.
    sub_scores[name] = {
        'data_sparse_gen':  _rescale(q1_auroc),
        'cross_domain_gen': _rescale(cd_auroc),
        'temporal_gen':     _rescale(tmp_auroc),
    }
    sub_scores_raw[name] = {
        'data_sparse_gen':  round(float(q1_auroc), 4)  if not np.isnan(q1_auroc)  else np.nan,
        'cross_domain_gen': round(float(cd_auroc), 4)  if not np.isnan(cd_auroc)  else np.nan,
        'temporal_gen':     round(float(tmp_auroc), 4) if not np.isnan(tmp_auroc) else np.nan,
        # Reported, not scored. Coverage and the full-denominator Hits@K sit
        # beside the scored AUROC so a reader can see how much of the gold set
        # each score rests on. They are deliberately not folded into
        # temporal_gen: coverage is already scored in dimension 01, and adding
        # it here would charge the same deficiency twice in Fig 1.
        'prospective_coverage':   _cov.get(name, {}).get('coverage', np.nan),
        'prospective_hits_all':   _cov.get(name, {}).get(f'hits@{HITS_K}_all', np.nan),
        'prospective_hits_eval':  _cov.get(name, {}).get(f'hits@{HITS_K}_evaluable', np.nan),
        'prospective_n_evaluable': _cov.get(name, {}).get('n_evaluable', 0),
        'prospective_n_gold':      N_GOLD,
    }

# Summary mean of available sub-scores (NaN sub-scores excluded)
summary_scores = {
    name: round(float(_safe_mean(list(ss.values()))), 4)
    for name, ss in sub_scores.items()
}

print('Sub-scores are rescaled to (AUROC - 0.5) / 0.5, matching notebook 06; '
      'raw AUROCs are kept in the checkpoint as sub_scores_raw.')

tier_scalars = {name: {tier: _scalars(m) for tier, m in tiers.items()}
                for name, tiers in tier_results_kg.items()}
area_scalars = {name: {area: _scalars(m) for area, m in areas.items()}
                for name, areas in area_results_kg.items()}
temporal_scalars = {name: _scalars(m) for name, m in temporal_metrics_kg.items()}

ckpt = {
    # Whether the prospective axis rests on the full gold standard or only on
    # the EMA/PMDA/TGA subset. Without this a reader of the checkpoint cannot
    # tell which of the two produced temporal_gen.
    'sub_scores_raw':   sub_scores_raw,
    'rescale_auroc':    RESCALE_AUROC,
    'fda_available':    bool(FDA_AVAILABLE),
    # Which gold standard produced temporal_gen, and which of the four checks were applied.
    'gold_standard_version':  GOLD_STANDARD_VERSION,
    'prospective_strict':     bool(PROSPECTIVE_STRICT),
    'prospective_common_support': bool(PROSPECTIVE_COMMON_SUPPORT),
    'prospective_filters':    prospective_filter_log,
    # The three-number view and the complete-case comparison, so notebook 00 and
    # the manuscript can report them without re-running this notebook.
    'prospective_coverage_table': prospective_coverage_df.to_dict('records'),
    'prospective_pairwise':       prospective_pairwise_df.to_dict('records'),
    'prospective_hits_k':         int(HITS_K),
    'prospective_eligible':       PROSPECTIVE_ELIGIBLE,
    'prospective_skipped':        prospective_skipped,
    'prospective_max_content_date': str(MAX_CONTENT_DATE.date()),
    'kg_content_dates':           {n: str(KG_SNAPSHOTS.get(n, {}).get('content_date'))
                                   for n in LOADED},
    'prospective_n_gold':         int(N_GOLD),
    'summary_scores':   summary_scores,
    'sub_scores':       sub_scores,
    'tier_scalars':     tier_scalars,
    'area_scalars':     area_scalars,
    'temporal_scalars': temporal_scalars,
    'area_cv':          {name: float(v) for name, v in area_cv.items()},
}

# Tables. 07 wrote no CSV at all, so every number in this dimension existed
# only inside a pickle and a figure: none of it could be recomputed or quoted
# from the results tree the way 04, 05 and 06 can be.
out.table(pd.DataFrame([{'KG': n, 'generalisation': summary_scores[n], **sub_scores[n]}
                        for n in LOADED]), 'dimension_scores')
out.table(pd.DataFrame([{'KG': n, 'tier': t, **m}
                        for n, tiers in tier_scalars.items()
                        for t, m in tiers.items()]), 'sparse_tiers')
out.table(pd.DataFrame([{'KG': n, 'area': a, **m}
                        for n, areas in area_scalars.items()
                        for a, m in areas.items()]), 'cross_domain')
out.table(pd.DataFrame([{'KG': n, **m} for n, m in temporal_scalars.items()]),
          'prospective')
out.table(pd.DataFrame([{'KG': n, 'auroc_cv_pct': v} for n, v in area_cv.items()]),
          'domain_equity')
out.checkpoint(ckpt)
print(f'\n{"KG":12s}  {"Sparse":>8s}  {"X-Domain":>8s}  {"Prospective":>11s}  {"Summary":>8s}')
print('-' * 56)
_fmt = lambda v: 'N/A' if (v is None or np.isnan(v)) else f'{v:.4f}'
for name in LOADED:
    ss = sub_scores[name]
    n_scored = sum(1 for v in ss.values() if not np.isnan(v))
    note = '' if n_scored == 3 else f'   ({n_scored}/3 axes scored)'
    print(f'  {name:12s}  {_fmt(ss["data_sparse_gen"]):>8s}  '
          f'{_fmt(ss["cross_domain_gen"]):>8s}  '
          f'{_fmt(ss["temporal_gen"]):>11s}  {_fmt(summary_scores[name]):>8s}{note}')
print()
# All three axes are AUROCs, so they share a 0.5 chance floor and the mean is on
# one scale. It is still a mean over whichever axes were scorable, so the count
# above says how many fed each summary.
print(out.summary())

Sub-scores are rescaled to (AUROC - 0.5) / 0.5, matching notebook 06; raw AUROCs are kept in the checkpoint as sub_scores_raw.

KG              Sparse  X-Domain  Prospective   Summary
--------------------------------------------------------
  primekg         0.3810    0.9166       0.5529    0.6168
  hetionet        0.5471    0.7549          N/A    0.6510   (2/3 axes scored)
  drkg            0.1158    0.8702       0.7358    0.5739
  openbilink      0.2841    0.5675       0.3537    0.4018
  biokg           0.2054    0.7763       0.6612    0.5476
  matrix          0.7500    0.9967          N/A    0.8734   (2/3 axes scored)

07_generalization: wrote 22 files
  results/checkpoints/07_generalization.pkl
  results/figures/07_generalization/07a_sparse_tiers.pdf
  results/figures/07_generalization/07a_sparse_tiers.png
  results/figures/07_generalization/07b_domain_heatmap.pdf
  results/figures/07_generalization/07b_domain_heatmap.png
  results/figures/07_generalization/07c_domain_cv.pdf
  resu

In [37]:
# --- figure-data cache -------------------------------------------------------
# Persist exactly the variables the figure functions need, so results/figures can
# be regenerated headless (tools/regenerate_figs.py) WITHOUT reloading the raw KGs.
#
# This cached the inputs to two of the seven figures. The other five were drawn
# inline in the notebook at the time, so they had no inputs to cache and could
# not be regenerated at all; now that all seven live in viz/dimension07.py they
# all need their inputs here or the headless path silently rebuilds a subset.

_figdata = {k: v for k, v in [
    # 07a sparse tiers
    ('tier_results_kg',     globals().get('tier_results_kg')),
    ('TIERS',               globals().get('TIERS')),
    # 07b domain heatmap, 07c domain CV
    ('area_results_kg',     globals().get('area_results_kg')),
    ('ALL_DOMAINS',         globals().get('ALL_DOMAINS')),
    ('area_cv',             globals().get('area_cv')),
    # 07d prospective AUROC, 07e hits@k
    ('baseline_all_kg',     globals().get('baseline_all_kg')),
    ('baseline_same_kg',    globals().get('baseline_same_kg')),
    ('temporal_metrics_kg', globals().get('temporal_metrics_kg')),
    # 07f rank distribution
    ('pct_data',            globals().get('pct_data')),
    # 07g network proximity
    ('all_proximity',       globals().get('all_proximity')),
    # shared
    ('LOADED',              globals().get('LOADED')),
    ('KG_COLORS',           globals().get('KG_COLORS')),
] if v is not None}

with open(CKPT_DIR / '07_figdata.pkl', 'wb') as _f:
    pickle.dump(_figdata, _f)
print('figure-data cache ->', '07_figdata.pkl', '| keys:', list(_figdata))


figure-data cache -> 07_figdata.pkl | keys: ['tier_results_kg', 'TIERS', 'area_results_kg', 'ALL_DOMAINS', 'area_cv', 'baseline_all_kg', 'baseline_same_kg', 'temporal_metrics_kg', 'pct_data', 'all_proximity', 'LOADED', 'KG_COLORS']


In [38]:
# Memory cleanup - free state before the next notebook in the pipeline.
_to_free = ['kgs', 'graphs', 'maps', 'preps',
            'kg_pos_pairs', 'kg_entity_sets', 'kg_disease_degree',
            'existing_ind_kg', 'temporal_pairs_kg']
for _v in _to_free:
    try:
        del globals()[_v]
    except KeyError:
        pass
gc.collect()
print('Freed KG state from kernel memory.')

Freed KG state from kernel memory.
